# 02_radiology_overlap_main

Main estimand:

- weighted counterfactual demographic shifts in model linear predictor (`beta`) by group
- hazard ratios (`HR`) by group
- group contrasts versus the reference level

Weighting layers:

1. ED-selection IPSW: from all ED stays into the radiology analytic cohort  
2. Group-overlap weights: baseline structured covariates or structured + chief complaint text

Heavy outputs are written under `./main/radiology/...` and are not recomputed when the target file already exists.

> `NOTE`: THERE IS A KNOWN BUG related to NumPy (see: https://github.com/numpy/numpy/issues/29820?timeline_page=1). This issue may trigger warnings during certain matrix operations (e.g., linear algebra routines). The warnings are known to be **harmless** and do **not** indicate numerical instability or incorrect results. In this pipeline, such warnings are intentionally **suppressed / ignored** to avoid unnecessary interruption or confusion.


In [1]:
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
import private_info
import warnings

from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

# Suppress known harmless sklearn/NumPy RuntimeWarnings from matmul overflow/divide-by-zero.
warnings.filterwarnings(
    "ignore",
    message=r".*encountered in matmul.*",
    category=RuntimeWarning,
)
warnings.filterwarnings(
    "ignore",
    category=RuntimeWarning,
    module=r"sklearn\.linear_model\._linear_loss",
)
warnings.filterwarnings(
    "ignore",
    category=RuntimeWarning,
    module=r"sklearn\.utils\.extmath",
)

ROOT = Path(".")
MAIN = ROOT / "main"
RAD_MAIN = MAIN / "radiology"
ASSET_DIR = MAIN / "shared" / "radiology"
for p in [RAD_MAIN, RAD_MAIN / "selection", RAD_MAIN / "main", RAD_MAIN / "bootstrap", RAD_MAIN / "robustness"]:
    p.mkdir(parents=True, exist_ok=True)

SEED = 2026
TAU = 6.0
TEXT_COL = "chiefcomplaint"

BASE = Path(private_info.path_to_data)
MIMIC = BASE / "mimiciv" / "3.1"
ED = BASE / "mimic-iv-ed" / "2.2"

TARGET_DIRS = {
    "hosp": MIMIC / "hosp",
    "ed": ED / "ed",
}

OUTCOME_TIME_COLS = [
    "time_to_any_rad_hours",
    "time_to_advanced_hours",
    "time_to_xray_hours",
]

REF = {
    "gender": "F",
    "race": "White",
    "language": "English",
}

GROUP_VARS = ["gender", "race", "language"]

radiology_analytic = pd.read_parquet(ASSET_DIR / "radiology_analytic_imputed.parquet")
radiology_text = pd.read_parquet(ASSET_DIR / "radiology_text_bundle.parquet")

z_cols = [c for c in radiology_text.columns if c.startswith("z_")]
radiology_analytic = radiology_analytic.reset_index(drop=True)
radiology_text = radiology_text.reset_index(drop=True)
for c in z_cols:
    radiology_analytic[c] = radiology_text[c].to_numpy()

radiology_analytic.head()

,subject_id,hadm_id,ed_stay_id,ed_intime,ed_outtime,gender,race,arrival_transport,disposition,temperature,...,z_10,z_11,z_12,z_13,z_14,z_15,z_16,z_17,z_18,z_19
0,10000032.0,22595853.0,33258284.0,2180-05-06 19:17:00,2180-05-06 23:30:00,F,White,AMBULANCE,ADMITTED,98.4,...,-0.027083,-0.109052,-0.050900,0.228835,-0.039130,-0.006038,0.001229,0.000066,0.002660,-0.000678
1,10000032.0,22841357.0,38112554.0,2180-06-26 15:54:00,2180-06-26 21:31:00,F,White,AMBULANCE,ADMITTED,98.9,...,-0.059831,-0.161745,-0.079485,0.309097,-0.049927,-0.005036,0.001431,-0.000519,0.001237,-0.001535
2,10000032.0,25742920.0,35968195.0,2180-08-05 20:58:00,2180-08-06 01:44:00,F,White,AMBULANCE,ADMITTED,99.4,...,0.043839,0.027357,0.020412,-0.003245,-0.004684,-0.005980,0.000306,0.001006,0.004353,0.001219
3,10000032.0,29079034.0,32952584.0,2180-07-22 16:24:00,2180-07-23 05:54:00,F,White,AMBULANCE,HOME,97.8,...,-0.001776,0.004413,0.002170,0.000281,0.002609,0.010464,0.016704,0.012979,0.000466,-0.002863
4,10000032.0,29079034.0,39399961.0,2180-07-23 05:54:00,2180-07-23 14:00:00,F,White,AMBULANCE,ADMITTED,98.7,...,-0.024462,-0.098724,-0.046029,0.206318,-0.035244,-0.005529,0.001072,0.000011,0.002395,-0.000597


In [2]:
def _to_str(s: pd.Series) -> pd.Series:
    return s.astype("string").str.strip()

def collapse_race(s: pd.Series) -> pd.Series:
    s = _to_str(s)

    def f(x):
        if pd.isna(x) or x == "":
            return "Unknown"
        u = str(x).upper()
        if "WHITE" in u:
            return "White"
        if "BLACK" in u:
            return "Black"
        if "HISPANIC" in u or "LATINO" in u:
            return "Hispanic/Latino"
        if "ASIAN" in u:
            return "Asian"
        if "PORTUGUESE" in u:
            return "Other"
        if "DECLINED" in u or "UNABLE" in u or "UNKNOWN" in u:
            return "Unknown"
        return "Other"

    return s.map(f)

def collapse_language(s: pd.Series, top_n=8, min_count=1000) -> pd.Series:
    s = _to_str(s)
    s = s.where(~(s.isna() | (s == "")), "Unknown")
    vc = s.value_counts()
    eligible = [x for x in vc.index if (vc[x] >= min_count and x != "Unknown")]
    keep = (["Unknown"] if "Unknown" in vc.index else []) + eligible[:top_n]
    return s.where(s.isin(keep), "Other")

def collapse_simple(s: pd.Series) -> pd.Series:
    x = _to_str(s)
    return x.where(~(x.isna() | (x == "")), "Unknown")

def handle_pain_critical(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    if "pain" not in df.columns:
        df["pain_critical"] = np.int8(0)
        return df

    pain_str = df["pain"].astype("string")
    mask_crit = pain_str.str.strip().str.lower().eq("critical")
    df["pain_critical"] = mask_crit.where(mask_crit.notna(), False).astype("int8")

    pain_num = pd.to_numeric(df["pain"], errors="coerce")
    pain_num = pain_num.where(~mask_crit, 10.0)
    df["pain"] = pain_num
    return df

def fill_chiefcomplaint_missing(df: pd.DataFrame, text_col: str = TEXT_COL, fill_value: str = "no_cc") -> pd.DataFrame:
    df = df.copy()
    raw = df[text_col].astype("string")
    stripped = raw.str.strip()
    missing = stripped.isna() | stripped.eq("")
    df["cc_missing"] = missing.astype("int8")
    df[text_col] = stripped.where(~missing, fill_value)
    return df

def build_survival_target(
    df: pd.DataFrame,
    time_col: str,
    follow_up_hours: float | str | pd.Series | np.ndarray | None = None,
):
    if {"surv_in_risk", "surv_duration_hours", "surv_event"}.issubset(df.columns):
        in_risk = pd.to_numeric(df["surv_in_risk"], errors="coerce").fillna(0).astype(bool)
        duration = pd.to_numeric(df.loc[in_risk, "surv_duration_hours"], errors="coerce").to_numpy(dtype=float)
        event = (
            pd.to_numeric(df.loc[in_risk, "surv_event"], errors="coerce")
            .fillna(0)
            .astype(np.int8)
            .to_numpy()
        )
        return in_risk.to_numpy(), duration.astype(float), event.astype(np.int8)

    t = pd.to_numeric(df[time_col], errors="coerce")
    active_at_t0 = t.notna() & (t <= 0)
    in_risk = ~active_at_t0
    t_risk = t.loc[in_risk]

    if follow_up_hours is None:
        fu_risk = pd.to_numeric(df.loc[in_risk, "follow_up_hours"], errors="coerce").to_numpy(dtype=float)
    elif isinstance(follow_up_hours, str):
        fu_risk = pd.to_numeric(df.loc[in_risk, follow_up_hours], errors="coerce").to_numpy(dtype=float)
    elif np.isscalar(follow_up_hours):
        fu_risk = np.repeat(float(follow_up_hours), int(in_risk.sum()))
    else:
        fu_arr = np.asarray(follow_up_hours, dtype=float)
        fu_risk = fu_arr[in_risk.to_numpy()]

    t_risk_arr = t_risk.to_numpy(dtype=float)
    event = np.isfinite(t_risk_arr) & (t_risk_arr > 0) & (t_risk_arr <= fu_risk)
    duration = np.where(event, t_risk_arr, fu_risk)
    return in_risk.to_numpy(), duration.astype(float), event.astype(np.int8)


def standardize_features(df: pd.DataFrame, cols: list[str]) -> pd.DataFrame:
    X = df[cols].copy()

    num_cols = [c for c in X.columns if pd.api.types.is_numeric_dtype(X[c])]
    cat_cols = [c for c in X.columns if c not in num_cols]

    parts = []

    if num_cols:
        X_num = pd.DataFrame(
            SimpleImputer(strategy="median", keep_empty_features=True).fit_transform(X[num_cols]),
            columns=num_cols,
            index=X.index,
        )
        X_num = pd.DataFrame(
            StandardScaler().fit_transform(X_num),
            columns=num_cols,
            index=X.index,
        )
        parts.append(X_num)

    if cat_cols:
        X_cat_raw = X[cat_cols].astype("string")
        X_cat_raw = pd.DataFrame(
            SimpleImputer(
                strategy="constant",
                fill_value="Unknown",
                keep_empty_features=True,
            ).fit_transform(X_cat_raw),
            columns=cat_cols,
            index=X.index,
        )
        X_cat = pd.get_dummies(X_cat_raw, dummy_na=False, drop_first=True).astype(float)
        parts.append(X_cat)

    return pd.concat(parts, axis=1)
    
def fit_binary_propensity(a, X, seed, sample_weight=None):
    model = LogisticRegression(
        penalty="l2",
        C=0.05,
        max_iter=2000,
        solver="lbfgs",
        random_state=seed,
    )
    model.fit(X, a, sample_weight=sample_weight)
    p = model.predict_proba(X)[:, 1]
    p = np.clip(p, 1e-6, 1 - 1e-6)
    return model, p

def fit_multiclass_propensity(a, X, seed, sample_weight=None):
    model = LogisticRegression(
        penalty="l2",
        C=0.05,
        max_iter=4000,
        solver="lbfgs",
        random_state=seed,
    )
    model.fit(X, a, sample_weight=sample_weight)
    p = model.predict_proba(X)
    p = np.clip(p, 1e-6, 1 - 1e-6)
    return model, p

def stabilized_group_weights(df, group_var, adjust_cols, seed, ref_map):
    keep = df[group_var].notna()
    d = df.loc[keep].copy()
    d["_row_id"] = d.index.to_numpy()
    d = d.reset_index(drop=True)
    ref = ref_map.get(group_var)

    a = d[group_var].astype("string").astype(str)
    X = standardize_features(d, adjust_cols)

    if a.nunique() == 2:
        levels = sorted(a.unique().tolist())
        target = (a == levels[1]).astype(int).to_numpy()
        _, p1 = fit_binary_propensity(target, X, seed=seed)
        prob_obs = np.where(target == 1, p1, 1.0 - p1)
        marg = target.mean()
        numer = np.where(target == 1, marg, 1.0 - marg)
        d["ipw_group"] = numer / prob_obs
        d["_group_level"] = a.to_numpy()
    else:
        levels = a.value_counts().index.tolist()
        codes = pd.Categorical(a, categories=levels).codes
        _, pm = fit_multiclass_propensity(codes, X, seed=seed)
        marg = a.value_counts(normalize=True).reindex(levels).to_numpy()
        numer = marg[codes]
        prob_obs = pm[np.arange(len(d)), codes]
        d["ipw_group"] = numer / prob_obs
        d["_group_level"] = a.to_numpy()

    if ref is None:
        ref = a.value_counts().index[0]

    if "selection_weight" not in d.columns:
        d["selection_weight"] = 1.0
    d["selection_weight"] = pd.to_numeric(d["selection_weight"], errors="coerce").astype(float)
    d["_group_ref"] = ref
    d["total_weight"] = d["selection_weight"] * d["ipw_group"]

    return d



## ED selection weights

In [3]:
patients = pd.read_csv(
        TARGET_DIRS["hosp"] / "patients.csv.gz")

In [4]:
def minimal_nonsense_qc(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    rad_rules = {
        "o2sat": (0.0, 100.0),
        "sbp": (1.0, 500.0),
        "dbp": (1.0, 400.0),
        "heartrate": (0.0, 500.0),
        "resprate": (0.0, 300.0),
        "temperature": (10.0, 200.0),
        "pain": (0.0, 10.0),
        "anchor_age": (0.0, 120.0),
    }

    if "temperature" in df.columns:
        t = pd.to_numeric(df["temperature"], errors="coerce")

        mask_c_like = t.notna() & (t >= 25.0) & (t <= 45.0)
        df["temperature_c_like"] = mask_c_like.astype("int8")
        df.loc[mask_c_like, "temperature"] = np.nan

        t2 = pd.to_numeric(df["temperature"], errors="coerce")
        bad_f = t2.notna() & ((t2 < 80.0) | (t2 > 110.0))
        df.loc[bad_f, "temperature"] = np.nan
    else:
        df["temperature_c_like"] = np.int8(0)

    for col, (lo, hi) in rad_rules.items():
        if col == "temperature":
            continue
        if col not in df.columns:
            continue
        x = pd.to_numeric(df[col], errors="coerce")
        bad = x.notna() & ((x < lo) | (x > hi))
        df.loc[bad, col] = np.nan

    return df

RAD_NUM_COVS = ["anchor_age", "acuity", "temperature", "heartrate", "resprate", "o2sat", "sbp", "dbp", "pain"]
def radiology_summary_qc(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    numeric_cols = [c for c in RAD_NUM_COVS if c in df.columns]
    if "anchor_age_sq" in df.columns:
        numeric_cols.append("anchor_age_sq")

    hard_bounds = {
        "anchor_age": (0.0, 120.0),
        "anchor_age_sq": (0.0, 120.0 ** 2),
        "acuity": (1.0, 5.0),
        "temperature": (80.0, 110.0),
        "heartrate": (10.0, 300.0),
        "resprate": (1.0, 80.0),
        "o2sat": (20.0, 100.0),
        "sbp": (20.0, 300.0),
        "dbp": (5.0, 250.0),
        "pain": (0.0, 10.0),
    }

    for c in numeric_cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    for c, (lo, hi) in hard_bounds.items():
        if c in df.columns:
            s = pd.to_numeric(df[c], errors="coerce")
            df[c] = s.where((s >= lo) & (s <= hi))

    for c in numeric_cols:
        s = pd.to_numeric(df[c], errors="coerce")
        vals = s.dropna()
        if len(vals) < 20:
            continue

        q1 = vals.quantile(0.25)
        q3 = vals.quantile(0.75)
        iqr = q3 - q1
        if not np.isfinite(iqr) or iqr <= 0:
            continue

        lo = q1 - 10.0 * iqr
        hi = q3 + 10.0 * iqr
        df[c] = s.where((s >= lo) & (s <= hi))

    return df


In [5]:
selection_base_path = RAD_MAIN / "selection" / "ed_selection_base.parquet"


if selection_base_path.exists():
    ed_selection_base = pd.read_parquet(selection_base_path)
else:
    triage = pd.read_csv(
        TARGET_DIRS["ed"] / "triage.csv.gz",
        usecols=[
            "stay_id",
            "chiefcomplaint",
            "temperature",
            "heartrate",
            "resprate",
            "o2sat",
            "sbp",
            "dbp",
            "pain",
            "acuity",
        ],
        dtype={"pain": "object"},
    )

    edstays = pd.read_csv(
        TARGET_DIRS["ed"] / "edstays.csv.gz",
        usecols=[
            "subject_id",
            "hadm_id",
            "stay_id",
            "intime",
            "outtime",
            "gender",
            "race",
            "arrival_transport",
        ],
    )

    patients = pd.read_csv(
        TARGET_DIRS["hosp"] / "patients.csv.gz",
        usecols=["subject_id", "anchor_age"],
    )

    ed_selection_base = (
        edstays
        .merge(triage, on="stay_id", how="left")
        .merge(patients, on="subject_id", how="left")
    )

    ed_selection_base["stay_id"] = pd.to_numeric(ed_selection_base["stay_id"], errors="coerce")
    ed_selection_base["subject_id"] = pd.to_numeric(ed_selection_base["subject_id"], errors="coerce")
    ed_selection_base["hadm_id"] = pd.to_numeric(ed_selection_base["hadm_id"], errors="coerce")

    ed_selection_base["selected_into_radiology"] = (
        ed_selection_base["stay_id"].isin(radiology_analytic["ed_stay_id"]).astype("int8")
    )

    ed_selection_base = handle_pain_critical(ed_selection_base)
    ed_selection_base = fill_chiefcomplaint_missing(ed_selection_base, TEXT_COL, "no_cc")

    ed_selection_base["gender"] = collapse_simple(ed_selection_base["gender"])
    ed_selection_base["race"] = collapse_simple(ed_selection_base["race"])
    ed_selection_base["arrival_transport"] = collapse_simple(ed_selection_base["arrival_transport"])
    ed_selection_base["anchor_age"] = pd.to_numeric(ed_selection_base["anchor_age"], errors="coerce")

    numeric_cols = [
        "anchor_age",
        "acuity",
        "temperature",
        "heartrate",
        "resprate",
        "o2sat",
        "sbp",
        "dbp",
        "pain",
    ]
    for c in numeric_cols:
        ed_selection_base[c] = pd.to_numeric(ed_selection_base[c], errors="coerce")

    ed_selection_base.to_parquet(selection_base_path, index=False)

ed_selection_base.head()

ed_selection_base = handle_pain_critical(ed_selection_base)
ed_selection_base = fill_chiefcomplaint_missing(ed_selection_base, TEXT_COL, "no_cc")
ed_selection_base = minimal_nonsense_qc(ed_selection_base)
ed_selection_base = radiology_summary_qc(ed_selection_base)

In [6]:
selection_covs = [
    "anchor_age",
    "gender",
    "acuity",
    "temperature",
    "heartrate",
    "resprate",
    "o2sat",
    "sbp",
    "dbp",
    "pain",
    "pain_critical",
    "arrival_transport",
    "cc_missing",
]

selection_weight_path = RAD_MAIN / "selection" / "selection_weights.parquet"

if selection_weight_path.exists():
    selection_weights = pd.read_parquet(selection_weight_path)
else:
    Xsel = standardize_features(ed_selection_base, selection_covs)
    ysel = ed_selection_base["selected_into_radiology"].to_numpy()
    selection_model, psel = fit_binary_propensity(ysel, Xsel, seed=SEED)
    marg = ysel.mean()
    sw = np.where(ysel == 1, marg / psel, (1.0 - marg) / (1.0 - psel))

    selection_weights = ed_selection_base[["subject_id", "stay_id", "selected_into_radiology"]].copy()
    selection_weights["selection_weight"] = sw
    selection_weights.to_parquet(selection_weight_path, index=False)
    joblib.dump(selection_model, RAD_MAIN / "selection" / "selection_model.joblib")

selection_weights.head()

,subject_id,stay_id,selected_into_radiology,selection_weight
0,10000032,33258284,1,0.898879
1,10000032,38112554,1,0.843875
2,10000032,35968195,1,0.690357
3,10000032,32952584,1,0.592198
4,10000032,39399961,1,0.607356


In [7]:
if "Xsel" not in locals():
    Xsel = standardize_features(ed_selection_base, selection_covs)

Xsel_np = Xsel.to_numpy(dtype=np.float64)

print("Xsel finite:", np.isfinite(Xsel_np).all())
print("Xsel max abs:", np.nanmax(np.abs(Xsel_np)))
print("psel finite:", np.isfinite(psel).all() if "psel" in locals() else "not computed in this session")

col_max = pd.Series(np.nanmax(np.abs(Xsel_np), axis=0), index=Xsel.columns)
print(col_max.sort_values(ascending=False).head(20))

for c in selection_covs:
    s = ed_selection_base[c]
    if pd.api.types.is_numeric_dtype(s):
        print(
            c,
            "finite =",
            np.isfinite(pd.to_numeric(s, errors="coerce")).all(),
            "min =",
            pd.to_numeric(s, errors="coerce").min(),
            "max =",
            pd.to_numeric(s, errors="coerce").max(),
        )
    else:
        print(c, s.astype("string").value_counts(dropna=False).head())

Xsel finite: True
Xsel max abs: 15.853346050127698
psel finite: True
o2sat                           15.853346
temperature                     12.235776
dbp                             11.762661
heartrate                        9.878227
resprate                         9.447110
sbp                              7.516641
acuity                           3.365373
anchor_age                       2.023524
pain                             1.405841
gender_M                         1.000000
arrival_transport_HELICOPTER     1.000000
arrival_transport_OTHER          1.000000
arrival_transport_UNKNOWN        1.000000
arrival_transport_WALK IN        1.000000
pain_critical                    0.000000
cc_missing                       0.000000
dtype: float64
anchor_age finite = False min = 18.0 max = 91.0
gender gender
F    229898
M    195189
Name: count, dtype: Int64
acuity finite = False min = 1.0 max = 5.0
temperature finite = False min = 86.6 max = 107.7
heartrate finite = False min = 10.0 max 

In [8]:
Xsel = standardize_features(ed_selection_base, selection_covs)

print("Xsel finite:", np.isfinite(Xsel.to_numpy(dtype=np.float64)).all())
print("Xsel max abs:", np.nanmax(np.abs(Xsel.to_numpy(dtype=np.float64))))
print(Xsel.abs().max().sort_values(ascending=False).head(20))

Xsel finite: True
Xsel max abs: 15.853346050127698
o2sat                           15.853346
temperature                     12.235776
dbp                             11.762661
heartrate                        9.878227
resprate                         9.447110
sbp                              7.516641
acuity                           3.365373
anchor_age                       2.023524
pain                             1.405841
gender_M                         1.000000
arrival_transport_HELICOPTER     1.000000
arrival_transport_OTHER          1.000000
arrival_transport_UNKNOWN        1.000000
arrival_transport_WALK IN        1.000000
pain_critical                    0.000000
cc_missing                       0.000000
dtype: float64


## Main weighted neural Cox contrasts

In [9]:
import importlib.util
import subprocess
import sys

missing = [pkg for pkg in ["pycox", "torchtuples"] if importlib.util.find_spec(pkg) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])

In [10]:

import hashlib
import json
import os
import pickle
import warnings

os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")

import joblib
import numpy as np
import pandas as pd
import torch
import torchtuples as tt
from pycox.evaluation import EvalSurv
from torch import nn
from tqdm.auto import tqdm

from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score, log_loss, r2_score
from sklearn.multioutput import MultiOutputRegressor
from sklearn.preprocessing import StandardScaler

# Suppress known harmless sklearn/NumPy RuntimeWarnings from matmul overflow/divide-by-zero.
warnings.filterwarnings(
    "ignore",
    message=r".*encountered in matmul.*",
    category=RuntimeWarning,
)
warnings.filterwarnings(
    "ignore",
    category=RuntimeWarning,
    module=r"sklearn\.linear_model\._linear_loss",
)
warnings.filterwarnings(
    "ignore",
    category=RuntimeWarning,
    module=r"sklearn\.utils\.extmath",
)

TITLE_FS = 30
LABEL_FS = 23
TICK_FS = 20

BOOT_B = 300
ROBUST_BOOT_B = 100
BLB_SUBSET_GAMMA = 0.7
BLB_S = 5
BLB_R = 30
VAL_FRAC = 0.2
EPOCHS = 256
PATIENCE = 30
LR = 1e-3
WEIGHT_DECAY = 1e-4
LAMBDA_BETA = 1e-4
LAMBDA_GAMMA = 5e-4
MAIN_TRIM_LABEL = "overlap"
MAIN_TRIM_Q = None

ROBUST_TRIM_SPECS = {}

BOOT_TOKEN_TOPK = 40
MULTIPLIER_BOOT_B = 2000
MULTIPLIER_BOOT_BATCH = 250
MIN_SUBSET_N = 80
MIN_SUBSET_EVENT = 20

def select_torch_device():
    if torch.backends.mps.is_available():
        return torch.device("mps")
    if torch.cuda.is_available():
        return torch.device("cuda")
    return torch.device("cpu")

device = select_torch_device()
print(f"torch device: {device}")

np.random.seed(SEED)
torch.manual_seed(SEED)
if device.type != "mps" and torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEMO_PREFIXES = ("gender_", "race_", "language_")
MODEL_METRIC_COLS = [
    "beta",
    "HR",
]

def stable_seed(*parts):
    payload = "||".join(map(str, parts)).encode("utf-8")
    return int(hashlib.blake2b(payload, digest_size=8).hexdigest(), 16) % (2**31 - 1)

def save_df_pair(df: pd.DataFrame, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_parquet(path, index=False)
    df.to_csv(path.with_suffix(".csv"), index=False)

def safe_nanquantile(x, q):
    x = np.asarray(x, dtype=float)
    x = x[np.isfinite(x)]
    if x.size == 0:
        return np.nan
    return float(np.quantile(x, q))

def safe_nanstat(x, fn, default=np.nan):
    x = np.asarray(x, dtype=float)
    x = x[np.isfinite(x)]
    if x.size == 0:
        return float(default)
    return float(fn(x))

def make_train_val_split(n: int, seed: int, val_frac: float = VAL_FRAC):
    rng = np.random.default_rng(seed)
    order = rng.permutation(n)
    n_val = max(1, int(np.floor(n * val_frac)))
    val_idx = np.sort(order[:n_val])
    train_idx = np.sort(order[n_val:])
    return train_idx, val_idx

def split_frequency_weight(freq_weight, val_frac, seed):
    rng = np.random.default_rng(seed)
    freq_weight = np.asarray(freq_weight, dtype=int)
    val_weight = rng.binomial(freq_weight, val_frac)
    train_weight = freq_weight - val_weight
    return train_weight.astype(float), val_weight.astype(float)

def normalize_string_series(s: pd.Series, unknown: str = "Unknown") -> pd.Series:
    s = s.astype("string")
    s = s.where(s.notna(), unknown)
    s = s.str.strip()
    s = s.mask(s == "", unknown)
    return s.astype(str)

def normalize_cc(s: pd.Series) -> pd.Series:
    return (
        s.astype("string")
        .str.lower()
        .str.replace(r"[^a-z0-9]+", " ", regex=True)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

def fit_feature_preprocessor(df: pd.DataFrame, cols: list[str]):
    X = df[cols].copy()
    num_cols = [c for c in cols if pd.api.types.is_numeric_dtype(X[c])]
    cat_cols = [c for c in cols if c not in num_cols]

    spec = {
        "cols": cols,
        "num_cols": num_cols,
        "cat_cols": cat_cols,
    }

    if num_cols:
        num_imputer = SimpleImputer(strategy="median", keep_empty_features=True)
        X_num = pd.DataFrame(
            num_imputer.fit_transform(X[num_cols]),
            columns=num_cols,
            index=X.index,
        )
        scaler = StandardScaler().fit(X_num)
        spec["num_imputer"] = num_imputer
        spec["scaler"] = scaler

    if cat_cols:
        cat_imputer = SimpleImputer(
            strategy="constant",
            fill_value="Unknown",
            keep_empty_features=True,
        )
        X_cat = pd.DataFrame(
            cat_imputer.fit_transform(X[cat_cols].astype("string")),
            columns=cat_cols,
            index=X.index,
        ).astype("string")

        cat_levels = {}
        for c in cat_cols:
            levels = sorted(pd.Index(X_cat[c].astype(str).unique()).tolist())

            if c in REF:
                ref_level = REF[c]
                levels = [lvl for lvl in levels if lvl != ref_level]
                levels = [ref_level] + levels

            cat_levels[c] = levels

        spec["cat_imputer"] = cat_imputer
        spec["cat_levels"] = cat_levels

    return spec

def transform_features(df: pd.DataFrame, spec) -> pd.DataFrame:
    parts = []

    if spec["num_cols"]:
        X_num = pd.DataFrame(
            spec["num_imputer"].transform(df[spec["num_cols"]]),
            columns=spec["num_cols"],
            index=df.index,
        )
        X_num = pd.DataFrame(
            spec["scaler"].transform(X_num),
            columns=spec["num_cols"],
            index=df.index,
        )
        parts.append(X_num.astype(np.float32))

    if spec["cat_cols"]:
        X_cat = pd.DataFrame(
            spec["cat_imputer"].transform(df[spec["cat_cols"]].astype("string")),
            columns=spec["cat_cols"],
            index=df.index,
        ).astype("string")

        dummy_parts = []
        for c in spec["cat_cols"]:
            levels = spec["cat_levels"][c]
            if len(levels) <= 1:
                continue

            cat = pd.Categorical(X_cat[c].astype(str), categories=levels)
            dummies = pd.get_dummies(
                cat,
                prefix=c,
                prefix_sep="_",
                drop_first=True,
            ).astype(np.float32)
            dummies.index = df.index
            dummy_parts.append(dummies)

        if dummy_parts:
            parts.append(pd.concat(dummy_parts, axis=1))

    if parts:
        return pd.concat(parts, axis=1)
    return pd.DataFrame(index=df.index)

def standardize_dense_block(df: pd.DataFrame, cols: list[str], train_idx: np.ndarray):
    X = df[cols].copy().astype(np.float32)
    mu = X.iloc[train_idx].mean(axis=0).to_numpy(dtype=np.float32)
    sd = X.iloc[train_idx].std(axis=0, ddof=0).to_numpy(dtype=np.float32)
    sd = np.where(sd == 0.0, 1.0, sd)
    Z = ((X.to_numpy(dtype=np.float32) - mu) / sd).astype(np.float32)
    return pd.DataFrame(Z, index=df.index, columns=cols), {"mu": mu, "sd": sd}

def get_demo_cols(X: pd.DataFrame, prefixes=DEMO_PREFIXES):
    return [c for c in X.columns if any(c.startswith(p) for p in prefixes)]

def split_group_and_level(term: str):
    for g in GROUP_VARS:
        prefix = f"{g}_"
        if term.startswith(prefix):
            return g, term[len(prefix):]
    return None, term

class AdditiveCoxNetText(nn.Module):
    def __init__(self, n_base: int, n_text: int, n_demo: int, hidden=(64, 32), dropout=0.1):
        super().__init__()
        self.n_base = int(n_base)
        self.n_text = int(n_text)
        self.n_demo = int(n_demo)

        layers = []
        in_dim = self.n_base
        for h in hidden:
            layers.append(nn.Linear(in_dim, int(h)))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(float(dropout)))
            in_dim = int(h)
        layers.append(nn.Linear(in_dim, 1))
        self.base_mlp = nn.Sequential(*layers)

        self.alpha = nn.Linear(self.n_demo, 1, bias=False)
        self.beta = nn.Linear(self.n_text, 1, bias=False)
        self.gamma = nn.Linear(self.n_demo, self.n_text, bias=False)

    def forward(self, x):
        xb = x[:, : self.n_base]
        z = x[:, self.n_base : self.n_base + self.n_text]
        d = x[:, self.n_base + self.n_text : self.n_base + self.n_text + self.n_demo]

        base_part = self.base_mlp(xb)
        demo_main = self.alpha(d)
        shared_text = self.beta(z)
        demo_text_int = (self.gamma(d) * z).sum(dim=1, keepdim=True)

        return base_part + demo_main + shared_text + demo_text_int

class StructuredCoxNet(nn.Module):
    def __init__(self, n_base: int, n_demo: int, hidden=(64, 32), dropout=0.1):
        super().__init__()
        self.n_base = int(n_base)
        self.n_demo = int(n_demo)

        layers = []
        in_dim = self.n_base
        for h in hidden:
            layers.append(nn.Linear(in_dim, int(h)))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(float(dropout)))
            in_dim = int(h)
        layers.append(nn.Linear(in_dim, 1))
        self.base_mlp = nn.Sequential(*layers)
        self.alpha = nn.Linear(self.n_demo, 1, bias=False)

    def forward(self, x):
        xb = x[:, : self.n_base]
        d = x[:, self.n_base : self.n_base + self.n_demo]
        return self.base_mlp(xb) + self.alpha(d)

def _binned_weighted_cox_loss(log_hz, durations, events, weights):
    order = torch.argsort(durations, descending=True)
    t = durations[order]
    e = events[order]
    w = weights[order]
    lp = log_hz[order].reshape(-1)

    if torch.sum(e) <= 0:
        return torch.zeros((), device=lp.device)

    shift = torch.max(lp.detach())
    risk_term = w * torch.exp(lp - shift)
    risk_cumsum = torch.cumsum(risk_term, dim=0)

    _, counts = torch.unique_consecutive(t, return_counts=True)
    ends = torch.cumsum(counts, dim=0) - 1
    group_id = torch.repeat_interleave(
        torch.arange(len(counts), device=lp.device),
        counts,
    )

    event_w = w * e
    group_event_w = torch.zeros(len(counts), device=lp.device, dtype=lp.dtype)
    group_event_lp = torch.zeros(len(counts), device=lp.device, dtype=lp.dtype)

    group_event_w.scatter_add_(0, group_id, event_w)
    group_event_lp.scatter_add_(0, group_id, event_w * lp)

    valid = group_event_w > 0
    denom = risk_cumsum[ends][valid]

    loss = -torch.sum(group_event_lp[valid])
    loss = loss + torch.sum(group_event_w[valid] * (torch.log(denom) + shift))

    total_event_weight = torch.sum(event_w)
    return loss / torch.clamp(total_event_weight, min=1e-8)

class WeightedCoxPHLoss(torch.nn.Module):
    def __init__(self, net=None, lambda_beta=0.0, lambda_gamma=0.0):
        super().__init__()
        self.net = net
        self.lambda_beta = float(lambda_beta)
        self.lambda_gamma = float(lambda_gamma)

    def forward(self, log_hz, durations, events, weights):
        loss = _binned_weighted_cox_loss(log_hz, durations, events, weights)

        penalty = torch.zeros((), device=log_hz.device)
        if self.net is not None and hasattr(self.net, "beta"):
            penalty = penalty + self.lambda_beta * torch.sum(self.net.beta.weight ** 2)
        if self.net is not None and hasattr(self.net, "gamma"):
            penalty = penalty + self.lambda_gamma * torch.sum(self.net.gamma.weight ** 2)

        return loss + penalty

class NeuralCoxModel:
    def __init__(self, net, model_kind: str):
        self.net = net.to(device)
        self.model_kind = model_kind
        self.baseline_event_times_ = None
        self.baseline_cumhaz_ = None
        self.train_info_ = {}
        self.tt_model = None

    def __getstate__(self):
        state = self.__dict__.copy()
        state["tt_model"] = None
        return state

    def __setstate__(self, state):
        self.__dict__.update(state)
        if "tt_model" not in self.__dict__:
            self.tt_model = None

    def predict(self, X):
        x = torch.as_tensor(np.asarray(X, dtype=np.float32), device=device)
        self.net.eval()
        with torch.no_grad():
            out = self.net(x).reshape(-1).detach().cpu().numpy()
        return out

    def compute_baseline_hazards(self, X, durations, events, sample_weight):
        lp = self.predict(X)
        t = np.asarray(durations, dtype=float)
        e = np.asarray(events, dtype=np.int8)
        w = np.asarray(sample_weight, dtype=float)

        mask = np.isfinite(t) & np.isfinite(w)
        t = t[mask]
        e = e[mask]
        w = w[mask]
        lp = lp[mask]

        event_times = np.sort(np.unique(t[e == 1]))
        if event_times.size == 0:
            self.baseline_event_times_ = np.array([], dtype=float)
            self.baseline_cumhaz_ = np.array([], dtype=float)
            return self

        exp_lp = np.exp(lp - np.max(lp))
        cumhaz = []
        running = 0.0
        for tt0 in event_times:
            fail = w[(t == tt0) & (e == 1)].sum()
            risk = np.sum(w[t >= tt0] * exp_lp[t >= tt0])
            dH = 0.0 if risk <= 0 else fail / risk
            running += dH
            cumhaz.append(running)

        shift = np.max(lp)
        self.baseline_event_times_ = event_times
        self.baseline_cumhaz_ = np.asarray(cumhaz, dtype=float) * np.exp(-shift)
        return self

    def predict_surv_df(self, X):
        lp = self.predict(X)
        if self.baseline_event_times_ is None:
            raise ValueError("Baseline hazards are not available.")
        if len(self.baseline_event_times_) == 0:
            return pd.DataFrame(np.ones((1, len(lp))), index=[0.0])

        rr = np.exp(lp)
        surv = np.exp(-np.outer(self.baseline_cumhaz_, rr))
        return pd.DataFrame(surv, index=self.baseline_event_times_)

def fit_weighted_neural_cox(
    X: pd.DataFrame,
    durations: np.ndarray,
    events: np.ndarray,
    sample_weight: np.ndarray | None = None,
    tr_idx: np.ndarray | None = None,
    val_idx: np.ndarray | None = None,
    train_weight: np.ndarray | None = None,
    val_weight: np.ndarray | None = None,
    model_kind: str = "structured",
    n_demo: int = 0,
    n_text: int = 0,
    hidden=(64, 32),
    dropout=0.1,
    lr: float = LR,
    weight_decay: float = WEIGHT_DECAY,
    epochs: int = EPOCHS,
    patience: int = PATIENCE,
    lambda_beta: float = LAMBDA_BETA,
    lambda_gamma: float = LAMBDA_GAMMA,
):
    X_np = X.to_numpy(np.float32)
    d_np = np.asarray(durations, dtype=np.float32)
    e_np = np.asarray(events, dtype=np.float32)

    if sample_weight is None:
        base_w = np.ones(len(X_np), dtype=np.float32)
    else:
        base_w = np.asarray(sample_weight, dtype=np.float32)

    if train_weight is None or val_weight is None:
        w_train = np.zeros(len(X_np), dtype=np.float32)
        w_val = np.zeros(len(X_np), dtype=np.float32)
        w_train[np.asarray(tr_idx, dtype=int)] = base_w[np.asarray(tr_idx, dtype=int)]
        w_val[np.asarray(val_idx, dtype=int)] = base_w[np.asarray(val_idx, dtype=int)]
    else:
        w_train = np.asarray(train_weight, dtype=np.float32)
        w_val = np.asarray(val_weight, dtype=np.float32)

    train_mask = w_train > 0
    val_mask = w_val > 0

    w_scale = float(np.mean(w_train[train_mask])) if train_mask.any() else 1.0
    if not np.isfinite(w_scale) or w_scale <= 0:
        w_scale = 1.0

    w_train = w_train / w_scale
    w_val = w_val / w_scale

    if model_kind == "structured":
        n_base = X_np.shape[1] - int(n_demo)
        net = StructuredCoxNet(n_base=n_base, n_demo=n_demo, hidden=hidden, dropout=dropout)
    else:
        n_base = X_np.shape[1] - int(n_demo) - int(n_text)
        net = AdditiveCoxNetText(
            n_base=n_base,
            n_text=n_text,
            n_demo=n_demo,
            hidden=hidden,
            dropout=dropout,
        )

    loss = WeightedCoxPHLoss(
        net=net,
        lambda_beta=lambda_beta,
        lambda_gamma=lambda_gamma,
    )
    tt_model = tt.Model(
        net,
        loss=loss,
        optimizer=tt.optim.Adam(lr=lr, weight_decay=weight_decay),
        device=device,
    )

    x_train = X_np[train_mask]
    y_train = (
        d_np[train_mask],
        e_np[train_mask],
        w_train[train_mask].astype(np.float32),
    )

    x_val = X_np[val_mask]
    y_val = (
        d_np[val_mask],
        e_np[val_mask],
        w_val[val_mask].astype(np.float32),
    )

    callbacks = [tt.cb.EarlyStopping(patience=int(patience))]
    log = tt_model.fit(
        x_train,
        y_train,
        batch_size=max(1, len(x_train)),
        epochs=int(epochs),
        callbacks=callbacks,
        verbose=False,
        shuffle=False,
        val_data=(x_val, y_val),
        val_batch_size=max(1, len(x_val)),
    )

    log_df = log.to_pandas()
    best_epoch = int(log_df["val_loss"].astype(float).idxmin())
    best_val_loss = float(log_df.loc[best_epoch, "val_loss"])

    model = NeuralCoxModel(net=net, model_kind=model_kind)
    model.tt_model = tt_model
    model.train_info_ = {
        "best_val_loss": best_val_loss,
        "best_epoch": best_epoch,
        "n_epoch_run": int(len(log_df)),
        "weight_mean_train": float(w_scale),
        "history_tail": log_df.tail(10).reset_index().rename(columns={"index": "epoch"}).to_dict("records"),
    }
    return model




def weighted_mean_scalar(x, w):
    x = np.asarray(x, dtype=float)
    w = np.asarray(w, dtype=float)
    mask = np.isfinite(x) & np.isfinite(w) & (w > 0)
    if not np.any(mask):
        return np.nan
    return float(np.sum(w[mask] * x[mask]) / np.sum(w[mask]))


def disparity_table_from_model_counterfactual(
    model: NeuralCoxModel,
    X: pd.DataFrame,
    tau: float,
    ref_map: dict,
    families=("gender_", "race_", "language_"),
    sample_weight=None,
):
    cols = X.columns.to_list()
    x_ref_template = X.copy()
    rows = []

    if sample_weight is None:
        sample_weight = np.ones(len(X), dtype=float)
    else:
        sample_weight = np.asarray(sample_weight, dtype=float)
        if len(sample_weight) != len(X):
            raise ValueError(f"sample_weight length={len(sample_weight)} does not match len(X)={len(X)}")

    for prefix in families:
        fam_cols = [c for c in cols if c.startswith(prefix)]
        if len(fam_cols) == 0:
            continue

        group_var = prefix[:-1]
        ref_level = ref_map.get(group_var)
        X_ref = x_ref_template.copy()
        X_ref.loc[:, fam_cols] = 0.0

        lp_ref = model.predict(X_ref)
        rows.append({
            "group_var": group_var,
            "level": ref_level,
            "ref_level": ref_level,
            "term": f"{group_var}_{ref_level}",
            "beta": 0.0,
            "HR": 1.0,
        })

        for fam_col in fam_cols:
            X_cf = X_ref.copy()
            X_cf.loc[:, fam_col] = 1.0
            lp_cf = model.predict(X_cf)
            beta = weighted_mean_scalar(lp_cf - lp_ref, sample_weight)
            level = fam_col.split("_", 1)[1]

            rows.append({
                "group_var": group_var,
                "level": level,
                "ref_level": ref_level,
                "term": fam_col,
                "beta": beta,
                "HR": float(np.exp(beta)) if np.isfinite(beta) else np.nan,
            })

    out = pd.DataFrame(rows)
    return out.sort_values(["group_var", "level"]).reset_index(drop=True)

def compare_disparity(tbl_left: pd.DataFrame, tbl_right: pd.DataFrame, left_scheme: str, right_scheme: str):
    join_cols = ["group_var", "level", "ref_level", "term"]

    for side_name, tbl in [("left", tbl_left), ("right", tbl_right)]:
        dup = tbl.loc[tbl.duplicated(join_cols, keep=False), join_cols + MODEL_METRIC_COLS]
        if len(dup) > 0:
            raise ValueError(
                f"Duplicate disparity rows detected on {side_name} side.\n"
                f"{dup.sort_values(join_cols).to_string(index=False)}"
            )

    left = tbl_left[join_cols + MODEL_METRIC_COLS].rename(
        columns={c: f"{c}_{left_scheme}" for c in MODEL_METRIC_COLS}
    )
    right = tbl_right[join_cols + MODEL_METRIC_COLS].rename(
        columns={c: f"{c}_{right_scheme}" for c in MODEL_METRIC_COLS}
    )

    out = left.merge(right, on=join_cols, how="inner")
    out["left_scheme"] = left_scheme
    out["right_scheme"] = right_scheme
    out["delta_beta"] = out[f"beta_{right_scheme}"] - out[f"beta_{left_scheme}"]
    out["HR_ratio"] = out[f"HR_{right_scheme}"] / out[f"HR_{left_scheme}"]
    out["abs_delta_beta"] = out["delta_beta"].abs()
    return out

def weighted_concordance_index(
    risk_score: np.ndarray,
    durations: np.ndarray,
    events: np.ndarray,
    sample_weight: np.ndarray | None = None,
):
    risk_score = np.asarray(risk_score, dtype=float).reshape(-1)
    durations = np.asarray(durations, dtype=float).reshape(-1)
    events = np.asarray(events, dtype=np.int8).reshape(-1)
    if sample_weight is None:
        sample_weight = np.ones_like(durations, dtype=float)
    else:
        sample_weight = np.asarray(sample_weight, dtype=float).reshape(-1)

    numer = 0.0
    denom = 0.0
    n = len(durations)
    for i in range(n):
        if events[i] != 1:
            continue
        at_risk = durations > durations[i]
        if not np.any(at_risk):
            continue
        pair_w = sample_weight[i] * sample_weight[at_risk]
        score_j = risk_score[at_risk]
        numer += np.sum(pair_w * (risk_score[i] > score_j))
        numer += 0.5 * np.sum(pair_w * (risk_score[i] == score_j))
        denom += np.sum(pair_w)

    if denom <= 0.0:
        return np.nan
    return float(numer / denom)


def cindex_bundle(model, X: pd.DataFrame, durations: np.ndarray, events: np.ndarray, sample_weight: np.ndarray):
    risk_score = np.asarray(model.predict(X), dtype=float).reshape(-1)
    return {
        "weighted": weighted_concordance_index(risk_score, durations, events, sample_weight=sample_weight),
        "unweighted": weighted_concordance_index(risk_score, durations, events, sample_weight=None),
    }



def get_family_weighting_input_df(
    df: pd.DataFrame,
    family_var: str,
    adjust_cols: list[str],
    freq_weight_col: str | None = None,
) -> pd.DataFrame:
    cols = [family_var] + [c for c in adjust_cols if c not in GROUP_VARS]
    if "selection_weight" in df.columns:
        cols.append("selection_weight")
    if freq_weight_col is not None and freq_weight_col in df.columns:
        cols.append(freq_weight_col)
    return df[list(dict.fromkeys(cols))].copy()


def fit_family_overlap_weights(df, family_var, adjust_cols, seed, ref_map, freq_weight_col=None):
    d = df.copy()
    d["_row_id"] = d.index.to_numpy()
    d = d.reset_index(drop=True)

    if freq_weight_col is None:
        freq = np.ones(len(d), dtype=float)
    else:
        freq = pd.to_numeric(d[freq_weight_col], errors="coerce").to_numpy(dtype=float)

    a = normalize_string_series(d[family_var], unknown="Unknown").astype(str)
    X = standardize_features(d, [c for c in adjust_cols if c not in GROUP_VARS])

    if a.nunique() == 2:
        levels = sorted(a.unique().tolist())
        target = (a == levels[1]).astype(int).to_numpy()
        model, p1 = fit_binary_propensity(
            target,
            X,
            seed=seed,
            sample_weight=freq,
        )
        pm = np.column_stack([1.0 - p1, p1])
        prob_obs = pm[np.arange(len(d)), target]
        class_labels = levels
        observed_code = target
    else:
        levels = a.value_counts().index.tolist()
        codes = pd.Categorical(a, categories=levels).codes
        model, pm = fit_multiclass_propensity(
            codes,
            X,
            seed=seed,
            sample_weight=freq,
        )
        prob_obs = pm[np.arange(len(d)), codes]
        class_labels = levels
        observed_code = codes

    overlap_h = 1.0 / np.sum(1.0 / pm, axis=1)

    d["family_var"] = family_var
    d["family_group"] = a.to_numpy()
    d["ipw_family"] = overlap_h / prob_obs
    if "selection_weight" not in d.columns:
        d["selection_weight"] = 1.0
    d["selection_weight"] = pd.to_numeric(d["selection_weight"], errors="coerce").astype(float)
    d["family_ref_label"] = ref_map.get(family_var)
    d["total_weight"] = d["selection_weight"] * d["ipw_family"]

    coef_rows = []
    feature_names = list(X.columns)
    coef = np.asarray(model.coef_, dtype=float)
    intercept = np.asarray(model.intercept_, dtype=float).reshape(-1)
    if coef.ndim == 1:
        coef = coef.reshape(1, -1)
    if len(class_labels) == 2 and coef.shape[0] == 1:
        row_iter = [(class_labels[1], coef[0], intercept[0])]
    else:
        row_iter = [(class_labels[k], coef[k], intercept[k]) for k in range(len(class_labels))]

    for class_label, coef_row, intercept_val in row_iter:
        for feature_name, coef_val in zip(feature_names, coef_row):
            coef_rows.append(
                {
                    "diagnostic_type": "family_weight_coef",
                    "family_var": family_var,
                    "class_label": class_label,
                    "feature": feature_name,
                    "coef": float(coef_val),
                    "intercept": float(intercept_val),
                }
            )

    diag = {
        "diagnostic_type": "family_weight_fit",
        "family_var": family_var,
        "n_rows": int(len(d)),
        "n_levels": int(len(class_labels)),
        "family_ref_label": ref_map.get(family_var),
        "family_weight_min": safe_nanstat(d["ipw_family"], np.min),
        "family_weight_p01": safe_nanquantile(d["ipw_family"], 0.01),
        "family_weight_p50": safe_nanquantile(d["ipw_family"], 0.50),
        "family_weight_p99": safe_nanquantile(d["ipw_family"], 0.99),
        "family_weight_max": safe_nanstat(d["ipw_family"], np.max),
        "family_prob_min": safe_nanstat(prob_obs, np.min),
        "family_prob_p01": safe_nanquantile(prob_obs, 0.01),
        "family_prob_p50": safe_nanquantile(prob_obs, 0.50),
        "family_prob_p99": safe_nanquantile(prob_obs, 0.99),
        "family_prob_max": safe_nanstat(prob_obs, np.max),
        "family_class_observed": int(pd.Series(observed_code).nunique()),
    }
    return {
        "weighted_df": d,
        "model": model,
        "diagnostics": diag,
        "coef_table": pd.DataFrame(coef_rows),
    }

def apply_weight_truncation(weighted_df: pd.DataFrame, trim_q):
    out = weighted_df.copy()
    if trim_q is None:
        return out
    lo, hi = trim_q
    w = out["total_weight"].to_numpy(dtype=float)
    q_lo = float(np.quantile(w, lo))
    q_hi = float(np.quantile(w, hi))
    out["total_weight"] = np.clip(w, q_lo, q_hi)
    return out


def fit_single_scheme_model(
    risk_df: pd.DataFrame,
    duration: np.ndarray,
    event: np.ndarray,
    structured_cols: list[str],
    text_cols: list[str] | None,
    tau: float,
    seed: int,
    scheme_name: str,
    family_var: str,
    family_weight_fit: dict,
    trim_q=None,
    freq_weight_col=None,
):
    weighted_df = apply_weight_truncation(family_weight_fit["weighted_df"], trim_q=trim_q)
    row_idx = weighted_df["_row_id"].to_numpy()
    fit_df = risk_df.iloc[row_idx].reset_index(drop=True).copy()
    duration_fit = np.asarray(duration[row_idx], dtype=float)
    event_fit = np.asarray(event[row_idx], dtype=np.int8)

    if freq_weight_col is None:
        freq_weight = np.ones(len(weighted_df), dtype=float)
    else:
        freq_weight = pd.to_numeric(weighted_df[freq_weight_col], errors="coerce").to_numpy(dtype=float)

    train_freq, val_freq = split_frequency_weight(freq_weight, VAL_FRAC, seed=seed)
    train_mask = train_freq > 0
    val_mask = val_freq > 0
    train_idx = np.flatnonzero(train_mask)
    val_idx = np.flatnonzero(val_mask)

    total_weight = weighted_df["total_weight"].to_numpy(dtype=float)
    total_weight_full = total_weight * freq_weight
    total_weight_train = total_weight * train_freq
    total_weight_val = total_weight * val_freq

    preproc = fit_feature_preprocessor(fit_df.iloc[train_idx].copy(), structured_cols)
    X_struct_all = transform_features(fit_df, preproc)
    demo_cols = get_demo_cols(X_struct_all)
    base_cols_no_demo = [c for c in X_struct_all.columns if c not in demo_cols]

    if text_cols is None:
        X_model = pd.concat(
            [X_struct_all[base_cols_no_demo], X_struct_all[demo_cols]],
            axis=1,
        ).astype(np.float32)
        model_kind = "structured"
        n_text = 0
    else:
        Z_std, z_spec = standardize_dense_block(fit_df, text_cols, train_idx)
        X_model = pd.concat(
            [X_struct_all[base_cols_no_demo], Z_std[text_cols], X_struct_all[demo_cols]],
            axis=1,
        ).astype(np.float32)
        model_kind = "text"
        n_text = len(text_cols)

    model = fit_weighted_neural_cox(
        X=X_model,
        durations=duration_fit,
        events=event_fit,
        sample_weight=total_weight_full,
        train_weight=total_weight_train,
        val_weight=total_weight_val,
        model_kind=model_kind,
        n_demo=len(demo_cols),
        n_text=n_text,
    )

    tbl = disparity_table_from_model_counterfactual(
        model=model,
        X=X_model,
        tau=tau,
        ref_map=REF,
        families=(f"{family_var}_",),
        sample_weight=total_weight_full,
    )
    tbl["scheme"] = scheme_name
    tbl["family_var"] = family_var
    tbl["n_risk"] = int(len(fit_df))
    tbl["n_event"] = int(event_fit.sum())
    tbl["weight_sum"] = float(np.sum(total_weight_full))

    alpha_demo = model.net.alpha.weight.detach().cpu().numpy().reshape(-1)
    if model_kind == "text":
        beta_z = model.net.beta.weight.detach().cpu().numpy().reshape(-1)
        gamma = model.net.gamma.weight.detach().cpu().numpy()
    else:
        beta_z = None
        gamma = None

    cindex_train = cindex_bundle(
        model=model,
        X=X_model.loc[train_mask].copy(),
        durations=duration_fit[train_mask],
        events=event_fit[train_mask],
        sample_weight=total_weight_train[train_mask],
    )
    cindex_val = cindex_bundle(
        model=model,
        X=X_model.loc[val_mask].copy(),
        durations=duration_fit[val_mask],
        events=event_fit[val_mask],
        sample_weight=total_weight_val[val_mask],
    )

    diagnostics = {
        "diagnostic_type": "model_fit",
        "group_var": family_var,
        "family_var": family_var,
        "scheme": scheme_name,
        "trim_label": "none" if trim_q is None else f"{trim_q[0]:.3f}_{trim_q[1]:.3f}",
        "n_rows": int(len(fit_df)),
        "n_event": int(event_fit.sum()),
        "n_demo_cols": int(len(demo_cols)),
        "n_text_cols": int(n_text),
        "weight_min": safe_nanstat(total_weight_full, np.min),
        "weight_p01": safe_nanquantile(total_weight_full, 0.01),
        "weight_p50": safe_nanquantile(total_weight_full, 0.50),
        "weight_p99": safe_nanquantile(total_weight_full, 0.99),
        "weight_max": safe_nanstat(total_weight_full, np.max),
        "effective_sample_size": float((np.sum(total_weight_full) ** 2) / np.sum(total_weight_full ** 2)),
        "best_val_loss": float(model.train_info_["best_val_loss"]),
        "best_epoch": int(model.train_info_["best_epoch"]),
        "n_epoch_run": int(model.train_info_["n_epoch_run"]),
        "cindex_train_weighted": cindex_train["weighted"],
        "cindex_train_unweighted": cindex_train["unweighted"],
        "cindex_val_weighted": cindex_val["weighted"],
        "cindex_val_unweighted": cindex_val["unweighted"],
    }

    artifact = {
        "model": model,
        "X_model": X_model,
        "fit_df": fit_df,
        "duration": duration_fit,
        "event": event_fit,
        "sample_weight": total_weight_full,
        "demo_cols": demo_cols,
        "text_cols": [] if text_cols is None else list(text_cols),
        "family_var": family_var,
        "alpha_demo": alpha_demo,
        "beta_z": beta_z,
        "gamma": gamma,
        "table": tbl.copy(),
        "preprocessor": preproc,
        "family_weight_fit": family_weight_fit,
    }
    return tbl, diagnostics, artifact


def fit_weighted_subset_compare(
    analytic_sub: pd.DataFrame,
    outcome_specs: list[dict],
    scheme_map: dict,
    left_scheme: str,
    right_scheme: str,
    seed_prefix: str,
    freq_weight_col=None,
):
    print(f"[fit_weighted_subset_compare] start seed_prefix={seed_prefix}")
    long_rows = []
    compare_rows = []
    count_rows = []
    diag_rows = []
    artifact_store = {}

    for outcome_spec in outcome_specs:
        time_col = outcome_spec["time_col"]
        tau = outcome_spec["tau"]
        print(f"[fit_weighted_subset_compare] outcome={time_col} stage=build_survival_target")
        in_risk, duration, event = build_survival_target(analytic_sub, time_col, tau)
        risk_df = analytic_sub.loc[in_risk].reset_index(drop=True).copy()
        print(
            f"[fit_weighted_subset_compare] outcome={time_col} stage=risk_set "
            f"n_risk={len(risk_df)} n_event={int(event.sum())}"
        )

        count_rows.append({
            "outcome": time_col,
            "tau": tau,
            "n_risk": int(len(risk_df)),
            "n_event": int(event.sum()),
        })

        artifact_store[time_col] = {}

        for family_var in GROUP_VARS:
            artifact_store[time_col][family_var] = {}
            scheme_tables = []

            for scheme_name, spec in scheme_map.items():
                print(
                    f"[fit_weighted_subset_compare] outcome={time_col} "
                    f"family={family_var} scheme={scheme_name} stage=family_weight"
                )
                adjust_cols_use = [c for c in spec["ipw_adjust_cols"] if c not in GROUP_VARS]
                family_weight_fit = fit_family_overlap_weights(
                    get_family_weighting_input_df(
                        risk_df,
                        family_var=family_var,
                        adjust_cols=adjust_cols_use,
                        freq_weight_col=freq_weight_col,
                    ),
                    family_var=family_var,
                    adjust_cols=adjust_cols_use,
                    seed=stable_seed(seed_prefix, time_col, family_var, scheme_name, "family_weight"),
                    ref_map=REF,
                    freq_weight_col=freq_weight_col,
                )

                family_diag = dict(family_weight_fit["diagnostics"])
                family_diag["outcome"] = time_col
                family_diag["tau"] = tau
                family_diag["scheme"] = scheme_name
                diag_rows.append(family_diag)

                coef_tbl = family_weight_fit["coef_table"].copy()
                if len(coef_tbl) > 0:
                    coef_tbl["outcome"] = time_col
                    coef_tbl["tau"] = tau
                    coef_tbl["scheme"] = scheme_name
                    diag_rows.extend(coef_tbl.to_dict("records"))

                print(
                    f"[fit_weighted_subset_compare] outcome={time_col} "
                    f"family={family_var} scheme={scheme_name} stage=model_fit"
                )
                tbl, diag, artifact = fit_single_scheme_model(
                    risk_df=risk_df,
                    duration=duration,
                    event=event,
                    structured_cols=spec["structured_cols"],
                    text_cols=spec.get("text_cols"),
                    tau=tau,
                    seed=stable_seed(seed_prefix, time_col, family_var, scheme_name, "model"),
                    scheme_name=scheme_name,
                    family_var=family_var,
                    family_weight_fit=family_weight_fit,
                    trim_q=spec.get("trim_q"),
                    freq_weight_col=freq_weight_col,
                )
                tbl["outcome"] = time_col
                tbl["tau"] = tau
                diag["outcome"] = time_col
                diag["tau"] = tau
                scheme_tables.append(tbl)
                diag_rows.append(diag)
                artifact_store[time_col][family_var][scheme_name] = artifact

            print(f"[fit_weighted_subset_compare] outcome={time_col} family={family_var} stage=compare")
            family_long = pd.concat(scheme_tables, ignore_index=True)
            long_rows.append(family_long)
            cmp = compare_disparity(
                tbl_left=family_long.loc[family_long["scheme"] == left_scheme].copy(),
                tbl_right=family_long.loc[family_long["scheme"] == right_scheme].copy(),
                left_scheme=left_scheme,
                right_scheme=right_scheme,
            )
            cmp["outcome"] = time_col
            cmp["tau"] = tau
            cmp["family_var"] = family_var
            compare_rows.append(cmp)

        print(f"[fit_weighted_subset_compare] outcome={time_col} stage=done")

    print(f"[fit_weighted_subset_compare] done seed_prefix={seed_prefix}")
    return (
        pd.concat(long_rows, ignore_index=True),
        pd.concat(compare_rows, ignore_index=True),
        pd.DataFrame(count_rows),
        pd.DataFrame(diag_rows),
        artifact_store,
    )

def summarize_shift_attenuation(raw_compare: pd.DataFrame, alt_compare: pd.DataFrame, alt_label: str):
    join_cols = ["outcome", "group_var", "level", "ref_level", "term"]
    raw = raw_compare[join_cols + ["delta_beta"]].rename(
        columns={
            "delta_beta": "delta_beta_raw",
        }
    )
    alt = alt_compare[join_cols + ["delta_beta"]].rename(
        columns={
            "delta_beta": f"delta_beta_{alt_label}",
        }
    )
    out = raw.merge(alt, on=join_cols, how="inner")

    out["abs_delta_beta_raw"] = out["delta_beta_raw"].abs()
    out[f"abs_delta_beta_{alt_label}"] = out[f"delta_beta_{alt_label}"].abs()

    out[f"{alt_label}_share_of_beta_shift"] = out[f"abs_delta_beta_{alt_label}"] / out["abs_delta_beta_raw"]
    out.loc[out["abs_delta_beta_raw"] == 0.0, f"{alt_label}_share_of_beta_shift"] = np.nan

    out[f"attenuation_of_beta_shift_{alt_label}"] = 1.0 - out[f"{alt_label}_share_of_beta_shift"]
    out[f"sign_same_beta_{alt_label}"] = np.sign(out["delta_beta_raw"]) == np.sign(out[f"delta_beta_{alt_label}"])
    return out

def summarize_attenuation_table(shift_df: pd.DataFrame, alt_label: str):
    return (
        shift_df
        .groupby(["outcome", "group_var"], as_index=False)
        .agg(
            n_levels=("level", "size"),
            median_abs_delta_beta_raw=("abs_delta_beta_raw", "median"),
            median_abs_delta_beta_alt=(f"abs_delta_beta_{alt_label}", "median"),
            median_alt_share_beta=(f"{alt_label}_share_of_beta_shift", "median"),
            median_attenuation_beta=(f"attenuation_of_beta_shift_{alt_label}", "median"),
            sign_same_beta_share=(f"sign_same_beta_{alt_label}", "mean"),
        )
    )

def flatten_columns(df: pd.DataFrame):
    out = df.copy()
    out.columns = [
        "_".join([str(x) for x in col if str(x) != ""]).strip("_")
        if isinstance(col, tuple) else col
        for col in out.columns.to_flat_index()
    ]
    return out

def build_transfer_point_outputs(transfer_compare: pd.DataFrame):
    transfer_shift = (
        transfer_compare[[
            "outcome",
            "group_var",
            "subgroup",
            "level",
            "ref_level",
            "term",
            "delta_beta",
        ]]
        .pivot(
            index=["outcome", "group_var", "level", "ref_level", "term"],
            columns="subgroup",
            values=["delta_beta"],
        )
        .reset_index()
    )
    transfer_shift = flatten_columns(transfer_shift)

    transfer_shift["abs_shift_beta_non_transfer"] = transfer_shift["delta_beta_non_transfer"].abs()
    transfer_shift["abs_shift_beta_transfer"] = transfer_shift["delta_beta_transfer"].abs()
    transfer_shift["transfer_over_non_beta_ratio"] = (
        transfer_shift["abs_shift_beta_transfer"] / transfer_shift["abs_shift_beta_non_transfer"]
    )
    transfer_shift.loc[transfer_shift["abs_shift_beta_non_transfer"] == 0.0, "transfer_over_non_beta_ratio"] = np.nan
    transfer_shift["sign_same_beta"] = (
        np.sign(transfer_shift["delta_beta_transfer"]) == np.sign(transfer_shift["delta_beta_non_transfer"])
    )

    transfer_shift_summary = (
        transfer_shift
        .groupby(["outcome", "group_var"], as_index=False)
        .agg(
            n_levels=("level", "size"),
            median_abs_shift_beta_non_transfer=("abs_shift_beta_non_transfer", "median"),
            median_abs_shift_beta_transfer=("abs_shift_beta_transfer", "median"),
            median_transfer_over_non_beta_ratio=("transfer_over_non_beta_ratio", "median"),
            same_direction_beta_share=("sign_same_beta", "mean"),
        )
    )
    return transfer_shift, transfer_shift_summary

def blb_ci_table(
    df: pd.DataFrame,
    group_cols: list[str],
    metric_cols: list[str],
    subset_col: str = "subset_iter",
    rep_col: str = "resample_iter",
) -> pd.DataFrame:
    per_subset_rows = []

    for keys, g in df.groupby([subset_col] + group_cols, dropna=False):
        if not isinstance(keys, tuple):
            keys = (keys,)

        subset_key = keys[0]
        value_keys = keys[1:]
        row = {subset_col: subset_key}

        for col, key in zip(group_cols, value_keys):
            row[col] = key

        row["blb_r"] = int(g[rep_col].nunique())

        for col in metric_cols:
            x = pd.to_numeric(g[col], errors="coerce")
            row[f"{col}_subset_mean"] = float(x.mean())
            row[f"{col}_subset_ci_low"] = float(x.quantile(0.025))
            row[f"{col}_subset_ci_high"] = float(x.quantile(0.975))

        per_subset_rows.append(row)

    per_subset = pd.DataFrame(per_subset_rows)

    rows = []
    for keys, g in per_subset.groupby(group_cols, dropna=False):
        if not isinstance(keys, tuple):
            keys = (keys,)

        row = {col: key for col, key in zip(group_cols, keys)}
        row["blb_s"] = int(g[subset_col].nunique())
        row["blb_r_mean"] = float(g["blb_r"].mean())

        for col in metric_cols:
            row[f"{col}_boot_mean"] = float(g[f"{col}_subset_mean"].mean())
            row[f"{col}_ci_low"] = float(g[f"{col}_subset_ci_low"].mean())
            row[f"{col}_ci_high"] = float(g[f"{col}_subset_ci_high"].mean())

        rows.append(row)

    return pd.DataFrame(rows)

def summarize_keyword_rank_stability(keyword_long: pd.DataFrame):
    if len(keyword_long) == 0:
        return pd.DataFrame()

    if "blb_draw_id" in keyword_long.columns:
        iter_col = "blb_draw_id"
    elif "bootstrap_iter" in keyword_long.columns:
        iter_col = "bootstrap_iter"
    elif {"subset_iter", "resample_iter"}.issubset(keyword_long.columns):
        keyword_long = keyword_long.copy()
        keyword_long["blb_draw_id"] = (
            keyword_long["subset_iter"].astype(str) + "::" + keyword_long["resample_iter"].astype(str)
        )
        iter_col = "blb_draw_id"
    else:
        iter_col = None

    out = (
        keyword_long
        .groupby(
            ["outcome", "group_var", "effect_group", "effect_level", "direction", "token"],
            as_index=False,
        )
        .agg(
            n_selected=(iter_col, "nunique") if iter_col is not None else ("rank", "size"),
            median_rank=("rank", "median"),
            mean_rank=("rank", "mean"),
            median_score=("score", "median"),
            sign_consistency=("score", lambda x: np.mean(np.sign(x) == np.sign(np.median(x)))),
        )
    )

    if iter_col is not None:
        out["selection_frequency"] = out["n_selected"] / keyword_long[iter_col].nunique()
    else:
        out["selection_frequency"] = np.nan

    return (
        out
        .sort_values(
            ["outcome", "group_var", "effect_group", "effect_level", "direction", "selection_frequency", "median_rank"],
            ascending=[True, True, True, True, True, False, True],
        )
        .reset_index(drop=True)
    )

def build_subject_subset(df: pd.DataFrame, subject_col: str, subset_subjects: np.ndarray):
    keep = df[subject_col].astype("string").isin(pd.Index(subset_subjects).astype("string"))
    return df.loc[keep].reset_index(drop=True).copy()

def attach_subject_frequency(
    df: pd.DataFrame,
    subject_col: str,
    subset_subjects: np.ndarray,
    counts: np.ndarray,
    out_col: str = "blb_freq_weight",
):
    w_map = pd.Series(
        np.asarray(counts, dtype=float),
        index=pd.Index(subset_subjects).astype("string"),
    )
    out = df.copy()
    out[out_col] = out[subject_col].astype("string").map(w_map).astype(float)
    return out

def run_cluster_blb_compare(
    analytic_sub: pd.DataFrame,
    outcome_specs: list[dict],
    scheme_map: dict,
    left_scheme: str,
    right_scheme: str,
    seed_prefix: str,
    subject_col: str = "subject_id",
    b_subject: int | None = None,
    s: int = 5,
    r: int = 30,
    vec=None,
    svd=None,
    demo_cols_ref_map: dict | None = None,
    min_subset_n: int = MIN_SUBSET_N,
    min_subset_event: int = MIN_SUBSET_EVENT,
    keyword_top_k: int = BOOT_TOKEN_TOPK,
    quiet_fit: bool = True,
    show_progress: bool = True,
    log_every_draw: bool = True,
):
    import contextlib
    import io
    from tqdm.auto import tqdm

    def screen_blb_sample(
        df: pd.DataFrame,
        subset_iter: int,
        resample_iter: int,
        draw_id: str,
        stage: str,
        n_subject_positive: int,
        n_rows_boot: int,
    ):
        rows = []
        failures = []

        for outcome_spec in outcome_specs:
            time_col = outcome_spec["time_col"]
            tau = outcome_spec["tau"]

            in_risk, _, event = build_survival_target(df, time_col, tau)
            n_risk = int(np.sum(in_risk))
            n_event = int(np.sum(event))

            row = {
                "outcome": time_col,
                "tau": tau,
                "subset_iter": int(subset_iter),
                "resample_iter": int(resample_iter),
                "blb_draw_id": draw_id,
                "screen_stage": stage,
                "n_risk": n_risk,
                "n_event": n_event,
                "n_subject_positive": int(n_subject_positive),
                "n_rows_boot": int(n_rows_boot),
                "min_subset_n": int(min_subset_n),
                "min_subset_event": int(min_subset_event),
                "screen_pass": bool((n_risk >= min_subset_n) and (n_event >= min_subset_event)),
            }
            rows.append(row)

            if n_risk < min_subset_n or n_event < min_subset_event:
                failures.append(f"{time_col}(n_risk={n_risk}, n_event={n_event})")

        return pd.DataFrame(rows), failures

    subjects = analytic_sub[subject_col].astype("string").drop_duplicates().to_numpy()
    G = len(subjects)

    if b_subject is None:
        b_subject = int(np.ceil(G ** BLB_SUBSET_GAMMA))

    compare_rows = []
    count_rows = []
    diag_rows = []
    keyword_rows = []
    param_lists = {}

    total_slots = int(s * r)
    total_attempted = 0
    total_success = 0
    total_skipped = 0
    total_subset_skipped = 0

    if show_progress:
        total_bar = tqdm(
            total=total_slots,
            desc=f"{seed_prefix} total",
            position=0,
            leave=True,
            dynamic_ncols=True,
            mininterval=0,
            smoothing=0,
            bar_format="{desc:<24} {percentage:3.0f}%|{bar:24}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}, {rate_fmt}]",
        )
    else:
        total_bar = None

    for j in range(s):
        rng_subset = np.random.default_rng(stable_seed(seed_prefix, "subset", j))
        subset_subjects = rng_subset.choice(subjects, size=b_subject, replace=False)

        subset_df = build_subject_subset(
            analytic_sub,
            subject_col=subject_col,
            subset_subjects=subset_subjects,
        )

        subset_draw_id = f"{j}::subset"
        subset_n_subject_positive = int(subset_df[subject_col].astype("string").nunique())
        subset_n_rows = int(len(subset_df))

        if show_progress:
            subset_bar = tqdm(
                total=r,
                desc=f"{seed_prefix} subset {j + 1}/{s}",
                position=1,
                leave=False,
                dynamic_ncols=True,
                mininterval=0,
                smoothing=0,
                bar_format="{desc:<24} {percentage:3.0f}%|{bar:24}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}, {rate_fmt}]",
            )
            subset_bar.set_postfix_str(
                f"rows={subset_n_rows} subjects={subset_n_subject_positive}",
                refresh=True,
            )
            tqdm.write(
                f"[{seed_prefix}] subset {j + 1}/{s} start | "
                f"subset_rows={subset_n_rows} subset_subjects={subset_n_subject_positive} "
                f"b_subject={b_subject}"
            )
        else:
            subset_bar = None

        subset_screen_df, subset_failures = screen_blb_sample(
            df=subset_df,
            subset_iter=j,
            resample_iter=-1,
            draw_id=subset_draw_id,
            stage="subset_screen",
            n_subject_positive=subset_n_subject_positive,
            n_rows_boot=subset_n_rows,
        )
        count_rows.append(subset_screen_df)

        if subset_failures:
            total_subset_skipped += 1
            diag_rows.append(pd.DataFrame([{
                "diag_type": "blb_screen",
                "screen_stage": "subset_screen",
                "subset_iter": int(j),
                "resample_iter": int(-1),
                "blb_draw_id": subset_draw_id,
                "blb_status": "skipped_subset",
                "skip_reason": " | ".join(subset_failures),
                "b_subject": int(b_subject),
                "subset_n_subject_positive": subset_n_subject_positive,
                "subset_n_rows": subset_n_rows,
                "min_subset_n": int(min_subset_n),
                "min_subset_event": int(min_subset_event),
            }]))

            if show_progress:
                total_bar.update(r)
                total_bar.set_postfix_str(
                    f"stage=subset_skip subset={j + 1}/{s} "
                    f"accepted={total_success} skipped={total_skipped} "
                    f"subset_skipped={total_subset_skipped}",
                    refresh=True,
                )
                tqdm.write(
                    f"[{seed_prefix}] subset {j + 1}/{s} skipped | "
                    f"{' | '.join(subset_failures)}"
                )
                subset_bar.close()
            continue

        subset_success = 0
        subset_skipped = 0

        for k in range(r):
            total_attempted += 1

            rng_resample = np.random.default_rng(
                stable_seed(seed_prefix, "subset", j, "resample", k)
            )
            counts = rng_resample.multinomial(G, np.repeat(1.0 / b_subject, b_subject))

            boot_df = attach_subject_frequency(
                subset_df,
                subject_col=subject_col,
                subset_subjects=subset_subjects,
                counts=counts,
                out_col="blb_freq_weight",
            )
            boot_df = boot_df.loc[boot_df["blb_freq_weight"] > 0].reset_index(drop=True)

            draw_id = f"{j}::{k}"
            n_subject_positive = int(boot_df[subject_col].astype("string").nunique())
            n_rows_boot = int(len(boot_df))

            if log_every_draw:
                tqdm.write(
                    f"[{seed_prefix}] subset {j + 1}/{s} resample {k + 1}/{r} screen | "
                    f"rows={n_rows_boot} subjects={n_subject_positive}"
                )

            if show_progress:
                total_bar.set_postfix_str(
                    f"stage=screen subset={j + 1}/{s} resample={k + 1}/{r} "
                    f"accepted={total_success} skipped={total_skipped} "
                    f"subset_skipped={total_subset_skipped} "
                    f"rows={n_rows_boot} subjects={n_subject_positive}",
                    refresh=True,
                )
                subset_bar.set_postfix_str(
                    f"stage=screen accepted={subset_success} skipped={subset_skipped} "
                    f"rows={n_rows_boot} subjects={n_subject_positive}",
                    refresh=True,
                )

            screen_df, draw_failures = screen_blb_sample(
                df=boot_df,
                subset_iter=j,
                resample_iter=k,
                draw_id=draw_id,
                stage="resample_screen",
                n_subject_positive=n_subject_positive,
                n_rows_boot=n_rows_boot,
            )
            count_rows.append(screen_df)

            if draw_failures:
                total_skipped += 1
                subset_skipped += 1

                diag_rows.append(pd.DataFrame([{
                    "diag_type": "blb_screen",
                    "screen_stage": "resample_screen",
                    "subset_iter": int(j),
                    "resample_iter": int(k),
                    "blb_draw_id": draw_id,
                    "blb_status": "skipped_resample",
                    "skip_reason": " | ".join(draw_failures),
                    "b_subject": int(b_subject),
                    "n_subject_positive": n_subject_positive,
                    "n_rows_boot": n_rows_boot,
                    "min_subset_n": int(min_subset_n),
                    "min_subset_event": int(min_subset_event),
                }]))

                if log_every_draw:
                    tqdm.write(
                        f"[{seed_prefix}] subset {j + 1}/{s} resample {k + 1}/{r} skipped | "
                        f"{' | '.join(draw_failures)}"
                    )

                if show_progress:
                    subset_bar.update(1)
                    subset_bar.set_postfix_str(
                        f"stage=skip accepted={subset_success} skipped={subset_skipped} "
                        f"rows={n_rows_boot} subjects={n_subject_positive}",
                        refresh=True,
                    )
                    total_bar.update(1)
                    total_bar.set_postfix_str(
                        f"stage=skip subset={j + 1}/{s} resample={k + 1}/{r} "
                        f"accepted={total_success} skipped={total_skipped} "
                        f"subset_skipped={total_subset_skipped}",
                        refresh=True,
                    )
                continue

            if log_every_draw:
                tqdm.write(
                    f"[{seed_prefix}] subset {j + 1}/{s} resample {k + 1}/{r} fit_start | "
                    f"rows={n_rows_boot} subjects={n_subject_positive}"
                )

            if show_progress:
                total_bar.set_postfix_str(
                    f"stage=fit subset={j + 1}/{s} resample={k + 1}/{r} "
                    f"accepted={total_success} skipped={total_skipped} "
                    f"subset_skipped={total_subset_skipped} "
                    f"rows={n_rows_boot} subjects={n_subject_positive}",
                    refresh=True,
                )
                subset_bar.set_postfix_str(
                    f"stage=fit accepted={subset_success} skipped={subset_skipped} "
                    f"rows={n_rows_boot} subjects={n_subject_positive}",
                    refresh=True,
                )

            fit_kwargs = dict(
                analytic_sub=boot_df,
                outcome_specs=outcome_specs,
                scheme_map=scheme_map,
                left_scheme=left_scheme,
                right_scheme=right_scheme,
                seed_prefix=f"{seed_prefix}_subset{j}_resample{k}",
                freq_weight_col="blb_freq_weight",
            )

            if quiet_fit:
                with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
                    _long, cmp, counts_df, diag, artifacts = fit_weighted_subset_compare(**fit_kwargs)
            else:
                _long, cmp, counts_df, diag, artifacts = fit_weighted_subset_compare(**fit_kwargs)

            cmp = cmp.copy()
            counts_df = counts_df.copy()
            diag = diag.copy()

            cmp["subset_iter"] = int(j)
            cmp["resample_iter"] = int(k)
            cmp["blb_draw_id"] = draw_id

            counts_df["subset_iter"] = int(j)
            counts_df["resample_iter"] = int(k)
            counts_df["blb_draw_id"] = draw_id
            counts_df["screen_stage"] = "fit_pass"
            counts_df["blb_status"] = "accepted"
            counts_df["n_subject_positive"] = n_subject_positive
            counts_df["n_rows_boot"] = n_rows_boot

            diag["subset_iter"] = int(j)
            diag["resample_iter"] = int(k)
            diag["blb_draw_id"] = draw_id
            diag["blb_status"] = "accepted"
            diag["resampling_method"] = "subject_level_cluster_blb"
            diag["b_subject"] = int(b_subject)
            diag["n_subject_positive"] = n_subject_positive
            diag["n_rows_boot"] = n_rows_boot

            compare_rows.append(cmp)
            count_rows.append(counts_df)
            diag_rows.append(diag)

            if vec is not None and svd is not None:
                boot_keywords = build_keyword_effect_tables(
                    artifacts,
                    vec,
                    svd,
                    top_k=keyword_top_k,
                )
                if len(boot_keywords) > 0:
                    boot_keywords = boot_keywords.copy()
                    boot_keywords["subset_iter"] = int(j)
                    boot_keywords["resample_iter"] = int(k)
                    boot_keywords["blb_draw_id"] = draw_id
                    keyword_rows.append(boot_keywords)

            for outcome, by_scheme in artifacts.items():
                if left_scheme not in by_scheme or right_scheme not in by_scheme:
                    continue

                left_artifact = by_scheme[left_scheme]
                right_artifact = by_scheme[right_scheme]

                if right_artifact["beta_z"] is None or right_artifact["gamma"] is None:
                    continue

                if demo_cols_ref_map is None:
                    demo_cols_ref = list(right_artifact["demo_cols"])
                else:
                    demo_cols_ref = list(demo_cols_ref_map.get(outcome, right_artifact["demo_cols"]))

                entry = param_lists.setdefault(
                    outcome,
                    {
                        "demo_cols": demo_cols_ref,
                        "alpha_demo_base_boot": [],
                        "alpha_demo_text_boot": [],
                        "beta_z_boot": [],
                        "gamma_boot": [],
                        "subset_iter": [],
                        "resample_iter": [],
                        "blb_draw_id": [],
                    },
                )

                entry["alpha_demo_base_boot"].append(
                    align_demo_vector_to_reference(
                        left_artifact["alpha_demo"],
                        left_artifact["demo_cols"],
                        demo_cols_ref,
                    )
                )
                entry["alpha_demo_text_boot"].append(
                    align_demo_vector_to_reference(
                        right_artifact["alpha_demo"],
                        right_artifact["demo_cols"],
                        demo_cols_ref,
                    )
                )
                entry["beta_z_boot"].append(
                    np.asarray(right_artifact["beta_z"], dtype=float).copy()
                )
                entry["gamma_boot"].append(
                    align_gamma_to_reference(
                        right_artifact["gamma"],
                        right_artifact["demo_cols"],
                        demo_cols_ref,
                    )
                )
                entry["subset_iter"].append(int(j))
                entry["resample_iter"].append(int(k))
                entry["blb_draw_id"].append(draw_id)

            total_success += 1
            subset_success += 1

            if log_every_draw:
                tqdm.write(
                    f"[{seed_prefix}] subset {j + 1}/{s} resample {k + 1}/{r} done | "
                    f"accepted={total_success} skipped={total_skipped}"
                )

            if show_progress:
                subset_bar.update(1)
                subset_bar.set_postfix_str(
                    f"stage=done accepted={subset_success} skipped={subset_skipped} "
                    f"rows={n_rows_boot} subjects={n_subject_positive}",
                    refresh=True,
                )
                total_bar.update(1)
                total_bar.set_postfix_str(
                    f"stage=done subset={j + 1}/{s} resample={k + 1}/{r} "
                    f"accepted={total_success} skipped={total_skipped} "
                    f"subset_skipped={total_subset_skipped}",
                    refresh=True,
                )

        if show_progress:
            subset_bar.close()
            tqdm.write(
                f"[{seed_prefix}] subset {j + 1}/{s} finish | "
                f"subset_accepted={subset_success} subset_skipped={subset_skipped} "
                f"total_success={total_success} total_skipped={total_skipped}"
            )

    if show_progress:
        total_bar.close()

    if len(compare_rows) == 0:
        raise ValueError(
            "No successful BLB draws. Increase b_subject or relax min_subset_n / min_subset_event."
        )

    blb_compare = pd.concat(compare_rows, ignore_index=True)
    blb_counts = pd.concat(count_rows, ignore_index=True) if len(count_rows) > 0 else pd.DataFrame()
    blb_diagnostics = pd.concat(diag_rows, ignore_index=True) if len(diag_rows) > 0 else pd.DataFrame()
    blb_keyword_long = pd.concat(keyword_rows, ignore_index=True) if len(keyword_rows) > 0 else pd.DataFrame()

    blb_param_store = {}
    for outcome, entry in param_lists.items():
        blb_param_store[outcome] = {
            "demo_cols": list(entry["demo_cols"]),
            "alpha_demo_base_boot": np.stack(entry["alpha_demo_base_boot"], axis=0),
            "alpha_demo_text_boot": np.stack(entry["alpha_demo_text_boot"], axis=0),
            "beta_z_boot": np.stack(entry["beta_z_boot"], axis=0),
            "gamma_boot": np.stack(entry["gamma_boot"], axis=0),
            "subset_iter": np.asarray(entry["subset_iter"], dtype=int),
            "resample_iter": np.asarray(entry["resample_iter"], dtype=int),
            "blb_draw_id": np.asarray(entry["blb_draw_id"], dtype=object),
        }

    print(
        f"[run_cluster_blb_compare] seed_prefix={seed_prefix} "
        f"attempted={total_attempted} success={total_success} "
        f"skipped={total_skipped} subset_skipped={total_subset_skipped}"
    )

    return blb_compare, blb_counts, blb_diagnostics, blb_keyword_long, blb_param_store

def load_text_assets(asset_dir: Path, prefix: str):
    vec = joblib.load(asset_dir / f"{prefix}_tfidf.joblib")
    svd = joblib.load(asset_dir / f"{prefix}_svd.joblib")
    return vec, svd

def orient_gamma_demo_by_text(gamma: np.ndarray, demo_cols: list[str]):
    gamma = np.asarray(gamma, dtype=float)
    n_demo = int(len(demo_cols))
    if gamma.ndim != 2:
        raise ValueError(f"gamma must be 2D, got shape={gamma.shape}")
    if gamma.shape[0] == n_demo:
        return gamma
    if gamma.shape[1] == n_demo:
        return gamma.T
    raise ValueError(f"gamma shape={gamma.shape} is incompatible with n_demo={n_demo}")


def align_demo_vector_to_reference(values, demo_cols_current: list[str], demo_cols_ref: list[str]):
    values = np.asarray(values, dtype=float).reshape(-1)
    out = np.zeros(len(demo_cols_ref), dtype=float)
    current_map = {col: idx for idx, col in enumerate(demo_cols_current)}
    for j_ref, col in enumerate(demo_cols_ref):
        if col in current_map:
            out[j_ref] = values[current_map[col]]
    return out


def align_gamma_to_reference(gamma, demo_cols_current: list[str], demo_cols_ref: list[str]):
    gamma = np.asarray(gamma, dtype=float)
    if gamma.ndim != 2:
        raise ValueError(f"gamma must be 2D, got shape={gamma.shape}")

    if gamma.shape[1] == len(demo_cols_current):
        gamma_text_by_demo = gamma
    elif gamma.shape[0] == len(demo_cols_current):
        gamma_text_by_demo = gamma.T
    else:
        raise ValueError(
            f"gamma shape={gamma.shape} is incompatible with demo_cols_current size={len(demo_cols_current)}"
        )

    out = np.zeros((gamma_text_by_demo.shape[0], len(demo_cols_ref)), dtype=float)
    current_map = {col: idx for idx, col in enumerate(demo_cols_current)}
    for j_ref, col in enumerate(demo_cols_ref):
        if col in current_map:
            out[:, j_ref] = gamma_text_by_demo[:, current_map[col]]
    return out


def make_point_store_compact_jointfit(
    artifact_store: dict,
    compare_df: pd.DataFrame,
    left_scheme: str = "structured",
    right_scheme: str = "structured_plus_text",
):
    compact = {}

    for outcome, by_scheme in artifact_store.items():
        if left_scheme not in by_scheme or right_scheme not in by_scheme:
            continue

        left_artifact = by_scheme[left_scheme]
        right_artifact = by_scheme[right_scheme]

        compact[outcome] = {
            "time_col": outcome,
            "demo_cols": list(right_artifact["demo_cols"]),
            "alpha_demo_base": np.asarray(left_artifact["alpha_demo"], dtype=float).copy(),
            "alpha_demo_text": np.asarray(right_artifact["alpha_demo"], dtype=float).copy(),
            "beta_z": None if right_artifact["beta_z"] is None else np.asarray(right_artifact["beta_z"], dtype=float).copy(),
            "gamma": None if right_artifact["gamma"] is None else np.asarray(right_artifact["gamma"], dtype=float).copy(),
            "tbl_base": left_artifact["table"].reset_index(drop=True).copy(),
            "tbl_text": right_artifact["table"].reset_index(drop=True).copy(),
            "compare": compare_df.loc[compare_df["outcome"] == outcome].reset_index(drop=True).copy(),
        }

    return compact



def build_keyword_effect_tables(artifact_store, vec, svd, top_k: int = BOOT_TOKEN_TOPK):
    vocab = np.asarray(vec.get_feature_names_out())
    components = np.asarray(svd.components_, dtype=float).T
    rows = []

    for outcome, by_family in artifact_store.items():
        for family_var, by_scheme in by_family.items():
            if "structured_plus_text" not in by_scheme:
                continue
            artifact = by_scheme["structured_plus_text"]
            beta_z = artifact["beta_z"]
            gamma = artifact["gamma"]
            demo_cols = artifact["demo_cols"]

            if beta_z is None or gamma is None:
                continue

            gamma_by_demo = orient_gamma_demo_by_text(gamma, demo_cols)

            effects = [(family_var, "shared", np.asarray(beta_z, dtype=float))]
            for j, demo_col in enumerate(demo_cols):
                demo_group, level = split_group_and_level(demo_col)
                if demo_group != family_var:
                    continue
                effect = np.asarray(beta_z, dtype=float) + gamma_by_demo[j]
                effects.append((demo_group, level, effect))

            for effect_group, effect_level, effect_vec in effects:
                token_scores = components @ np.asarray(effect_vec, dtype=float)

                pos_idx = np.argsort(-token_scores)[:top_k]
                neg_idx = np.argsort(token_scores)[:top_k]

                for rank, idx in enumerate(pos_idx, start=1):
                    rows.append({
                        "outcome": outcome,
                        "family_var": family_var,
                        "group_var": effect_group,
                        "effect_group": effect_group,
                        "effect_level": effect_level,
                        "direction": "positive",
                        "rank": rank,
                        "token": str(vocab[idx]),
                        "score": float(token_scores[idx]),
                    })
                for rank, idx in enumerate(neg_idx, start=1):
                    rows.append({
                        "outcome": outcome,
                        "family_var": family_var,
                        "group_var": effect_group,
                        "effect_group": effect_group,
                        "effect_level": effect_level,
                        "direction": "negative",
                        "rank": rank,
                        "token": str(vocab[idx]),
                        "score": float(token_scores[idx]),
                    })

    return pd.DataFrame(rows)

def summarize_keyword_rank_stability(keyword_long: pd.DataFrame):
    if len(keyword_long) == 0:
        return pd.DataFrame()

    if "blb_draw_id" in keyword_long.columns:
        iter_col = "blb_draw_id"
    elif "bootstrap_iter" in keyword_long.columns:
        iter_col = "bootstrap_iter"
    elif {"subset_iter", "resample_iter"}.issubset(keyword_long.columns):
        keyword_long = keyword_long.copy()
        keyword_long["blb_draw_id"] = (
            keyword_long["subset_iter"].astype(str) + "::" + keyword_long["resample_iter"].astype(str)
        )
        iter_col = "blb_draw_id"
    else:
        iter_col = None

    out = (
        keyword_long
        .groupby(
            ["outcome", "group_var", "effect_group", "effect_level", "direction", "token"],
            as_index=False,
        )
        .agg(
            n_selected=(iter_col, "nunique") if iter_col is not None else ("rank", "size"),
            median_rank=("rank", "median"),
            mean_rank=("rank", "mean"),
            median_score=("score", "median"),
            sign_consistency=("score", lambda x: np.mean(np.sign(x) == np.sign(np.median(x)))),
        )
    )

    if iter_col is not None:
        out["selection_frequency"] = out["n_selected"] / keyword_long[iter_col].nunique()
    else:
        out["selection_frequency"] = np.nan

    return (
        out
        .sort_values(
            ["outcome", "group_var", "effect_group", "effect_level", "direction", "selection_frequency", "median_rank"],
            ascending=[True, True, True, True, True, False, True],
        )
        .reset_index(drop=True)
    )

def make_demo_residualized_text_features(
    df: pd.DataFrame,
    text_cols: list[str],
    demo_cols: list[str],
    train_idx: np.ndarray,
    seed: int,
):
    feature_spec = fit_feature_preprocessor(df.iloc[train_idx].copy(), demo_cols)
    X_train = transform_features(df.iloc[train_idx], feature_spec)
    X_all = transform_features(df, feature_spec)

    E = df[text_cols].to_numpy(np.float32)

    base = HistGradientBoostingRegressor(
        max_depth=3,
        learning_rate=0.05,
        max_iter=400,
        random_state=seed,
        early_stopping=False,
    )
    residualizer = MultiOutputRegressor(base, n_jobs=-1)
    residualizer.fit(X_train.to_numpy(np.float32), E[train_idx])

    E_hat = residualizer.predict(X_all.to_numpy(np.float32)).astype(np.float32)
    E_res = (E - E_hat).astype(np.float32)

    z_res_cols = [f"{c}__resid" for c in text_cols]
    E_res_df = pd.DataFrame(E_res, index=df.index, columns=z_res_cols)

    return E_res_df, {
        "r2_mean": float(np.mean(r2_score(E, E_hat, multioutput="raw_values"))),
        "r2_varw": float(r2_score(E, E_hat, multioutput="variance_weighted")),
    }

def make_conditional_residualized_text_features(
    df: pd.DataFrame,
    text_cols: list[str],
    structured_cols: list[str],
    demo_cols: list[str],
    train_idx: np.ndarray,
    seed: int,
):
    structured_cols = list(dict.fromkeys(structured_cols))
    structured_demo_cols = list(dict.fromkeys(structured_cols + demo_cols))

    spec_s = fit_feature_preprocessor(df.iloc[train_idx].copy(), structured_cols)
    spec_sd = fit_feature_preprocessor(df.iloc[train_idx].copy(), structured_demo_cols)

    XS_train = transform_features(df.iloc[train_idx], spec_s)
    XS_all = transform_features(df, spec_s)
    XSD_train = transform_features(df.iloc[train_idx], spec_sd)
    XSD_all = transform_features(df, spec_sd)

    E = df[text_cols].to_numpy(np.float32)

    base_s = HistGradientBoostingRegressor(
        max_depth=3,
        learning_rate=0.05,
        max_iter=400,
        random_state=seed,
        early_stopping=False,
    )
    base_sd = HistGradientBoostingRegressor(
        max_depth=3,
        learning_rate=0.05,
        max_iter=400,
        random_state=seed + 1,
        early_stopping=False,
    )

    residualizer_s = MultiOutputRegressor(base_s, n_jobs=-1)
    residualizer_sd = MultiOutputRegressor(base_sd, n_jobs=-1)

    residualizer_s.fit(XS_train.to_numpy(np.float32), E[train_idx])
    residualizer_sd.fit(XSD_train.to_numpy(np.float32), E[train_idx])

    E_hat_s = residualizer_s.predict(XS_all.to_numpy(np.float32)).astype(np.float32)
    E_hat_sd = residualizer_sd.predict(XSD_all.to_numpy(np.float32)).astype(np.float32)
    E_cond = (E - E_hat_sd + E_hat_s).astype(np.float32)

    z_cond_cols = [f"{c}__condresid" for c in text_cols]
    E_cond_df = pd.DataFrame(E_cond, index=df.index, columns=z_cond_cols)

    return E_cond_df, {
        "r2_structured_only_mean": float(np.mean(r2_score(E, E_hat_s, multioutput="raw_values"))),
        "r2_structured_only_varw": float(r2_score(E, E_hat_s, multioutput="variance_weighted")),
        "r2_structured_plus_demo_mean": float(np.mean(r2_score(E, E_hat_sd, multioutput="raw_values"))),
        "r2_structured_plus_demo_varw": float(r2_score(E, E_hat_sd, multioutput="variance_weighted")),
    }

def evaluate_demographic_predictability(
    df: pd.DataFrame,
    feature_cols: list[str],
    target_col: str,
    train_idx: np.ndarray,
    val_idx: np.ndarray,
):
    feature_spec = fit_feature_preprocessor(df.iloc[train_idx].copy(), feature_cols)
    X_train = transform_features(df.iloc[train_idx], feature_spec)
    X_val = transform_features(df.iloc[val_idx], feature_spec)

    y_train = normalize_string_series(df.iloc[train_idx][target_col]).reset_index(drop=True)
    y_val = normalize_string_series(df.iloc[val_idx][target_col]).reset_index(drop=True)

    labels = sorted(set(y_train.tolist()) & set(y_val.tolist()))
    train_mask = y_train.isin(labels).to_numpy()
    val_mask = y_val.isin(labels).to_numpy()

    clf = LogisticRegression(max_iter=4000, C=0.05)
    clf.fit(X_train.to_numpy(np.float32)[train_mask], y_train.to_numpy()[train_mask])

    proba = clf.predict_proba(X_val.to_numpy(np.float32)[val_mask])
    pred = clf.predict(X_val.to_numpy(np.float32)[val_mask])

    return {
        "n_train": int(train_mask.sum()),
        "n_val": int(val_mask.sum()),
        "log_loss": float(log_loss(y_val.to_numpy()[val_mask], proba, labels=clf.classes_)),
        "balanced_acc": float(balanced_accuracy_score(y_val.to_numpy()[val_mask], pred)),
    }

def get_full_compare_ref(compare_results: pd.DataFrame):
    return compare_results[[
        "outcome",
        "group_var",
        "level",
        "ref_level",
        "term",
        "delta_beta",
    ]].rename(
        columns={
            "delta_beta": "delta_beta_full",
        }
    )

def build_objective_point_outputs(objective_compare: pd.DataFrame, objective_counts: pd.DataFrame, full_compare_ref: pd.DataFrame):
    objective_vs_full = objective_compare.merge(
        full_compare_ref,
        on=["outcome", "group_var", "level", "ref_level", "term"],
        how="left",
    )

    objective_vs_full["abs_shift_objective_beta"] = objective_vs_full["delta_beta"].abs()
    objective_vs_full["abs_shift_full_beta"] = objective_vs_full["delta_beta_full"].abs()
    objective_vs_full["objective_over_full_beta_ratio"] = (
        objective_vs_full["abs_shift_objective_beta"] / objective_vs_full["abs_shift_full_beta"]
    )
    objective_vs_full.loc[objective_vs_full["abs_shift_full_beta"] == 0.0, "objective_over_full_beta_ratio"] = np.nan

    objective_vs_full["same_direction_beta_vs_full"] = (
        np.sign(objective_vs_full["delta_beta"]) == np.sign(objective_vs_full["delta_beta_full"])
    )

    objective_summary = (
        objective_vs_full
        .groupby(["outcome", "group_var"], as_index=False)
        .agg(
            n_levels=("level", "size"),
            median_abs_shift_objective_beta=("abs_shift_objective_beta", "median"),
            median_abs_shift_full_beta=("abs_shift_full_beta", "median"),
            median_objective_over_full_beta_ratio=("objective_over_full_beta_ratio", "median"),
            same_direction_beta_vs_full_share=("same_direction_beta_vs_full", "mean"),
        )
        .merge(objective_counts, on="outcome", how="left")
    )
    return objective_vs_full, objective_summary


def collect_counterfactual_beta_contributions(
    model: NeuralCoxModel,
    X: pd.DataFrame,
    ref_map: dict,
    families=DEMO_PREFIXES,
):
    cols = X.columns.to_list()
    x_ref_template = X.copy()
    contrib = {}
    meta_rows = []

    for prefix in families:
        fam_cols = [c for c in cols if c.startswith(prefix)]
        if len(fam_cols) == 0:
            continue

        group_var = prefix[:-1]
        ref_level = ref_map.get(group_var)
        X_ref = x_ref_template.copy()
        X_ref.loc[:, fam_cols] = 0.0
        lp_ref = np.asarray(model.predict(X_ref), dtype=float).reshape(-1)

        ref_term = f"{group_var}_{ref_level}"
        contrib[ref_term] = np.zeros(len(X_ref), dtype=float)
        meta_rows.append({
            "group_var": group_var,
            "level": ref_level,
            "ref_level": ref_level,
            "term": ref_term,
        })

        for fam_col in fam_cols:
            X_cf = X_ref.copy()
            X_cf.loc[:, fam_col] = 1.0
            lp_cf = np.asarray(model.predict(X_cf), dtype=float).reshape(-1)
            level = fam_col.split("_", 1)[1]
            contrib[fam_col] = lp_cf - lp_ref
            meta_rows.append({
                "group_var": group_var,
                "level": level,
                "ref_level": ref_level,
                "term": fam_col,
            })

    meta = pd.DataFrame(meta_rows)
    if len(meta) == 0:
        return contrib, meta
    meta = meta.drop_duplicates(["group_var", "level", "ref_level", "term"])
    return contrib, meta.sort_values(["group_var", "level", "term"]).reset_index(drop=True)

def make_multiplier_point_store_jointfit(
    artifact_store: dict,
    compare_df: pd.DataFrame,
    left_scheme: str = "structured",
    right_scheme: str = "structured_plus_text",
    subject_col: str = "subject_id",
):
    point_store = {}

    for outcome, by_scheme in artifact_store.items():
        if left_scheme not in by_scheme or right_scheme not in by_scheme:
            continue

        left_artifact = by_scheme[left_scheme]
        right_artifact = by_scheme[right_scheme]

        left_contrib, _ = collect_counterfactual_beta_contributions(
            model=left_artifact["model"],
            X=left_artifact["X_model"],
            ref_map=REF,
        )
        right_contrib, _ = collect_counterfactual_beta_contributions(
            model=right_artifact["model"],
            X=right_artifact["X_model"],
            ref_map=REF,
        )

        compare_sub = compare_df.loc[compare_df["outcome"] == outcome].reset_index(drop=True).copy()
        if len(compare_sub) == 0:
            continue

        fit_df = right_artifact["fit_df"].reset_index(drop=True).copy()
        subject_ids = fit_df[subject_col].astype("string").to_numpy()
        subject_codes, subject_levels = pd.factorize(subject_ids, sort=False)

        cluster_n = np.bincount(subject_codes, minlength=len(subject_levels)).astype(float)
        n_rows = int(cluster_n.sum())

        term_order = compare_sub["term"].astype(str).tolist()

        base_row_mat = np.column_stack([
            np.asarray(left_contrib.get(term, np.zeros(n_rows, dtype=float)), dtype=float)
            for term in term_order
        ])
        text_row_mat = np.column_stack([
            np.asarray(right_contrib.get(term, np.zeros(n_rows, dtype=float)), dtype=float)
            for term in term_order
        ])
        delta_row_mat = text_row_mat - base_row_mat

        cluster_base_sum = np.zeros((len(subject_levels), len(term_order)), dtype=float)
        cluster_text_sum = np.zeros((len(subject_levels), len(term_order)), dtype=float)
        cluster_delta_sum = np.zeros((len(subject_levels), len(term_order)), dtype=float)

        np.add.at(cluster_base_sum, subject_codes, base_row_mat)
        np.add.at(cluster_text_sum, subject_codes, text_row_mat)
        np.add.at(cluster_delta_sum, subject_codes, delta_row_mat)

        point_base = cluster_base_sum.sum(axis=0) / float(n_rows)
        point_text = cluster_text_sum.sum(axis=0) / float(n_rows)
        point_delta = cluster_delta_sum.sum(axis=0) / float(n_rows)

        compare_point = compare_sub.copy()
        compare_point["beta_structured_plugin"] = point_base
        compare_point["beta_structured_plus_text_plugin"] = point_text
        compare_point["delta_beta_plugin"] = point_delta

        compare_point["HR_structured_plugin"] = np.exp(point_base)
        compare_point["HR_structured_plus_text_plugin"] = np.exp(point_text)
        compare_point["HR_ratio_plugin"] = np.exp(point_delta)

        compare_point["beta_structured_plugin_error"] = (
            compare_point["beta_structured_plugin"]
            - pd.to_numeric(compare_point["beta_structured"], errors="coerce")
        )
        compare_point["beta_structured_plus_text_plugin_error"] = (
            compare_point["beta_structured_plus_text_plugin"]
            - pd.to_numeric(compare_point["beta_structured_plus_text"], errors="coerce")
        )
        compare_point["delta_beta_plugin_error"] = (
            compare_point["delta_beta_plugin"]
            - pd.to_numeric(compare_point["delta_beta"], errors="coerce")
        )

        compare_point["HR_structured_plugin_error"] = (
            compare_point["HR_structured_plugin"]
            - pd.to_numeric(compare_point["HR_structured"], errors="coerce")
        )
        compare_point["HR_structured_plus_text_plugin_error"] = (
            compare_point["HR_structured_plus_text_plugin"]
            - pd.to_numeric(compare_point["HR_structured_plus_text"], errors="coerce")
        )
        compare_point["HR_ratio_plugin_error"] = (
            compare_point["HR_ratio_plugin"]
            - pd.to_numeric(compare_point["HR_ratio"], errors="coerce")
        )

        point_store[outcome] = {
            "compare_point": compare_point,
            "cluster_n": cluster_n,
            "cluster_base_sum": cluster_base_sum,
            "cluster_text_sum": cluster_text_sum,
            "cluster_delta_sum": cluster_delta_sum,
            "n_rows": int(n_rows),
            "n_clusters": int(len(subject_levels)),
        }

    return point_store


def run_subject_multiplier_bootstrap(
    point_store: dict,
    seed_prefix: str,
    n_boot: int = MULTIPLIER_BOOT_B,
    batch_size: int = MULTIPLIER_BOOT_BATCH,
):
    compare_rows = []
    diagnostics_rows = []

    for outcome, payload in point_store.items():
        cluster_n = np.asarray(payload["cluster_n"], dtype=float)
        cluster_base_sum = np.asarray(payload["cluster_base_sum"], dtype=float)
        cluster_text_sum = np.asarray(payload["cluster_text_sum"], dtype=float)
        cluster_delta_sum = np.asarray(payload["cluster_delta_sum"], dtype=float)
        compare_point = payload["compare_point"].reset_index(drop=True).copy()

        n_clusters = int(payload["n_clusters"])
        n_rows = int(payload["n_rows"])

        diagnostics_rows.append({
            "diagnostic_type": "subject_multiplier_bootstrap",
            "outcome": outcome,
            "n_clusters": n_clusters,
            "n_rows": n_rows,
            "n_boot": int(n_boot),
            "max_abs_beta_structured_plugin_error": float(
                np.nanmax(np.abs(pd.to_numeric(compare_point["beta_structured_plugin_error"], errors="coerce")))
            ),
            "max_abs_beta_structured_plus_text_plugin_error": float(
                np.nanmax(np.abs(pd.to_numeric(compare_point["beta_structured_plus_text_plugin_error"], errors="coerce")))
            ),
            "max_abs_delta_beta_plugin_error": float(
                np.nanmax(np.abs(pd.to_numeric(compare_point["delta_beta_plugin_error"], errors="coerce")))
            ),
            "max_abs_HR_structured_plugin_error": float(
                np.nanmax(np.abs(pd.to_numeric(compare_point["HR_structured_plugin_error"], errors="coerce")))
            ),
            "max_abs_HR_structured_plus_text_plugin_error": float(
                np.nanmax(np.abs(pd.to_numeric(compare_point["HR_structured_plus_text_plugin_error"], errors="coerce")))
            ),
            "max_abs_HR_ratio_plugin_error": float(
                np.nanmax(np.abs(pd.to_numeric(compare_point["HR_ratio_plugin_error"], errors="coerce")))
            ),
        })

        rng = np.random.default_rng(stable_seed(seed_prefix, outcome, "subject_multiplier"))
        boot_iter = 0

        for start in range(0, n_boot, batch_size):
            m = int(min(batch_size, n_boot - start))
            multiplier = rng.exponential(scale=1.0, size=(m, n_clusters))
            denom = multiplier @ cluster_n

            base_draw = (multiplier @ cluster_base_sum) / denom[:, None]
            text_draw = (multiplier @ cluster_text_sum) / denom[:, None]
            delta_draw = (multiplier @ cluster_delta_sum) / denom[:, None]

            for b in range(m):
                draw_df = compare_point.copy()
                draw_df["bootstrap_iter"] = int(boot_iter)

                draw_df["beta_structured"] = base_draw[b]
                draw_df["beta_structured_plus_text"] = text_draw[b]
                draw_df["delta_beta"] = delta_draw[b]

                draw_df["HR_structured"] = np.exp(base_draw[b])
                draw_df["HR_structured_plus_text"] = np.exp(text_draw[b])
                draw_df["HR_ratio"] = np.exp(delta_draw[b])

                draw_df["abs_delta_beta"] = np.abs(delta_draw[b])

                compare_rows.append(draw_df)
                boot_iter += 1

    compare_boot = pd.concat(compare_rows, ignore_index=True) if compare_rows else pd.DataFrame()
    diagnostics = pd.DataFrame(diagnostics_rows)
    return compare_boot, diagnostics

def multiplier_ci_table(
    df: pd.DataFrame,
    group_cols: list[str],
    metric_cols: list[str],
    rep_col: str = "bootstrap_iter",
) -> pd.DataFrame:
    rows = []

    for keys, g in df.groupby(group_cols, dropna=False):
        if not isinstance(keys, tuple):
            keys = (keys,)

        row = {col: key for col, key in zip(group_cols, keys)}
        row["multiplier_boot_b"] = int(g[rep_col].nunique())

        for col in metric_cols:
            x = pd.to_numeric(g[col], errors="coerce")
            row[f"{col}_boot_mean"] = float(x.mean())
            row[f"{col}_ci_low"] = float(x.quantile(0.025))
            row[f"{col}_ci_high"] = float(x.quantile(0.975))

        rows.append(row)

    return pd.DataFrame(rows)


torch device: mps


In [11]:

base_adjust_cols = [
    "anchor_age",
    "anchor_age_sq",
    "acuity",
    "temperature",
    "heartrate",
    "resprate",
    "o2sat",
    "sbp",
    "dbp",
    "pain",
    "pain_critical",
    "temperature_c_like",
    "cc_missing",
    "arrival_transport",
    "insurance",
]

text_adjust_cols = list(dict.fromkeys(base_adjust_cols + z_cols))

analytic_main = radiology_analytic.merge(
    selection_weights.loc[
        selection_weights["selected_into_radiology"] == 1,
        ["stay_id", "selection_weight"],
    ].rename(columns={"stay_id": "ed_stay_id"}),
    on="ed_stay_id",
    how="left",
)
analytic_main["selection_weight"] = analytic_main["selection_weight"].where(
    analytic_main["selection_weight"].notna(),
    1.0,
)

OUTCOME_SPECS = [{"time_col": c, "tau": TAU} for c in OUTCOME_TIME_COLS]
MAIN_SCHEME_MAP = {
    "structured": {
        "structured_cols": GROUP_VARS + base_adjust_cols,
        "text_cols": None,
        "ipw_adjust_cols": base_adjust_cols,
        "trim_q": MAIN_TRIM_Q,
    },
    "structured_plus_text": {
        "structured_cols": GROUP_VARS + base_adjust_cols,
        "text_cols": z_cols,
        "ipw_adjust_cols": text_adjust_cols,
        "trim_q": MAIN_TRIM_Q,
    },
}

vec, svd = load_text_assets(ASSET_DIR, "radiology")


In [12]:
import time

main_results_path = RAD_MAIN / "main" / "radiology_weighted_main.parquet"
compare_results_path = RAD_MAIN / "main" / "radiology_weighted_compare.parquet"
main_counts_path = RAD_MAIN / "main" / "radiology_weighted_counts.parquet"
main_diag_path = RAD_MAIN / "main" / "radiology_weighted_diagnostics.parquet"
main_keyword_path = RAD_MAIN / "main" / "radiology_keyword_reverse_engineering.parquet"
main_artifact_path = RAD_MAIN / "main" / "radiology_main_artifacts.pkl"

t0_total = time.perf_counter()
print("[main] start")

all_exist = all(
    p.exists()
    for p in [
        main_results_path,
        compare_results_path,
        main_counts_path,
        main_diag_path,
        main_keyword_path,
        main_artifact_path,
    ]
)

if all_exist:
    print("[main] cache hit -> loading saved outputs")

    t0 = time.perf_counter()
    main_results = pd.read_parquet(main_results_path)
    print(f"[main] loaded main_results in {time.perf_counter() - t0:.2f}s")

    t0 = time.perf_counter()
    compare_results = pd.read_parquet(compare_results_path)
    print(f"[main] loaded compare_results in {time.perf_counter() - t0:.2f}s")

    t0 = time.perf_counter()
    main_counts = pd.read_parquet(main_counts_path)
    print(f"[main] loaded main_counts in {time.perf_counter() - t0:.2f}s")

    t0 = time.perf_counter()
    main_diagnostics = pd.read_parquet(main_diag_path)
    print(f"[main] loaded main_diagnostics in {time.perf_counter() - t0:.2f}s")

    t0 = time.perf_counter()
    main_keyword_reverse_engineering = pd.read_parquet(main_keyword_path)
    print(f"[main] loaded main_keyword_reverse_engineering in {time.perf_counter() - t0:.2f}s")

    t0 = time.perf_counter()
    with open(main_artifact_path, "rb") as f:
        main_artifacts = pickle.load(f)
    print(f"[main] loaded main_artifacts in {time.perf_counter() - t0:.2f}s")

else:
    print("[main] cache miss -> recomputing outputs")

    t0 = time.perf_counter()
    main_results, compare_results, main_counts, main_diagnostics, main_artifacts = fit_weighted_subset_compare(
        analytic_sub=analytic_main.reset_index(drop=True).copy(),
        outcome_specs=OUTCOME_SPECS,
        scheme_map=MAIN_SCHEME_MAP,
        left_scheme="structured",
        right_scheme="structured_plus_text",
        seed_prefix="radiology_main",
    )
    print(f"[main] fit_weighted_subset_compare finished in {time.perf_counter() - t0:.2f}s")

    t0 = time.perf_counter()
    main_keyword_reverse_engineering = build_keyword_effect_tables(main_artifacts, vec, svd, top_k=50)
    print(f"[main] build_keyword_effect_tables finished in {time.perf_counter() - t0:.2f}s")

    t0 = time.perf_counter()
    save_df_pair(main_results, main_results_path)
    print(f"[main] saved main_results in {time.perf_counter() - t0:.2f}s")

    t0 = time.perf_counter()
    save_df_pair(compare_results, compare_results_path)
    print(f"[main] saved compare_results in {time.perf_counter() - t0:.2f}s")

    t0 = time.perf_counter()
    save_df_pair(main_counts, main_counts_path)
    print(f"[main] saved main_counts in {time.perf_counter() - t0:.2f}s")

    t0 = time.perf_counter()
    save_df_pair(main_diagnostics, main_diag_path)
    print(f"[main] saved main_diagnostics in {time.perf_counter() - t0:.2f}s")

    t0 = time.perf_counter()
    save_df_pair(main_keyword_reverse_engineering, main_keyword_path)
    print(f"[main] saved main_keyword_reverse_engineering in {time.perf_counter() - t0:.2f}s")

    t0 = time.perf_counter()
    with open(main_artifact_path, "wb") as f:
        pickle.dump(main_artifacts, f)
    print(f"[main] saved main_artifacts in {time.perf_counter() - t0:.2f}s")

print(f"[main] total elapsed = {time.perf_counter() - t0_total:.2f}s")

main_results.head()
compare_results.head()
main_keyword_reverse_engineering.head()

[main] start
[main] cache miss -> recomputing outputs
[fit_weighted_subset_compare] start seed_prefix=radiology_main
[fit_weighted_subset_compare] outcome=time_to_any_rad_hours stage=build_survival_target
[fit_weighted_subset_compare] outcome=time_to_any_rad_hours stage=risk_set n_risk=203016 n_event=117540
[fit_weighted_subset_compare] outcome=time_to_any_rad_hours family=gender scheme=structured stage=family_weight
[fit_weighted_subset_compare] outcome=time_to_any_rad_hours family=gender scheme=structured stage=model_fit
[fit_weighted_subset_compare] outcome=time_to_any_rad_hours family=gender scheme=structured_plus_text stage=family_weight
[fit_weighted_subset_compare] outcome=time_to_any_rad_hours family=gender scheme=structured_plus_text stage=model_fit
[fit_weighted_subset_compare] outcome=time_to_any_rad_hours family=gender stage=compare
[fit_weighted_subset_compare] outcome=time_to_any_rad_hours family=race scheme=structured stage=family_weight
[fit_weighted_subset_compare] out

,outcome,family_var,group_var,effect_group,effect_level,direction,rank,token,score
0,time_to_any_rad_hours,gender,gender,gender,shared,positive,1,chest pain,0.067471
1,time_to_any_rad_hours,gender,gender,gender,shared,positive,2,chest,0.067422
2,time_to_any_rad_hours,gender,gender,gender,shared,positive,3,fall,0.066149
3,time_to_any_rad_hours,gender,gender,gender,shared,positive,4,fever,0.035677
4,time_to_any_rad_hours,gender,gender,gender,shared,positive,5,dizziness,0.030433


## bootstrap by `subject_id`

In [13]:
import json
import pickle
from pathlib import Path

import joblib
import pandas as pd

EXPORT_DIR = RAD_MAIN / "bootstrap_state_bundle"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

compare_results.to_parquet(EXPORT_DIR / "compare_results.parquet", index=False)

with open(EXPORT_DIR / "main_artifacts.pkl", "wb") as f:
    pickle.dump(main_artifacts, f, protocol=pickle.HIGHEST_PROTOCOL)

joblib.dump(vec, EXPORT_DIR / "vec.joblib")
joblib.dump(svd, EXPORT_DIR / "svd.joblib")

with open(EXPORT_DIR / "REF.pkl", "wb") as f:
    pickle.dump(REF, f, protocol=pickle.HIGHEST_PROTOCOL)

config = {
    "VAL_FRAC": VAL_FRAC,
    "LR": LR,
    "WEIGHT_DECAY": WEIGHT_DECAY,
    "EPOCHS": EPOCHS,
    "PATIENCE": PATIENCE,
    "LAMBDA_BETA": LAMBDA_BETA,
    "LAMBDA_GAMMA": LAMBDA_GAMMA,
    "LEFT_SCHEME": "structured",
    "RIGHT_SCHEME": "structured_plus_text",
    "SUBJECT_COL": "subject_id",
    "FULL_BOOT_N_ITER": 300,
}

with open(EXPORT_DIR / "config.json", "w", encoding="utf-8") as f:
    json.dump(config, f, ensure_ascii=False, indent=2)

print("Saved bundle to:", EXPORT_DIR)

Saved bundle to: main/radiology/bootstrap_state_bundle


## Robustness

### weight trimming

In [14]:
robustness_path = RAD_MAIN / "robustness" / "weight_trimming.parquet"
robustness_summary_path = RAD_MAIN / "robustness" / "weight_trimming_summary.parquet"

weight_trimming_results = pd.DataFrame()
weight_trimming_summary = pd.DataFrame()

save_df_pair(weight_trimming_results, robustness_path)
save_df_pair(weight_trimming_summary, robustness_summary_path)

weight_trimming_summary

""


In [15]:
print(weight_trimming_summary)

Empty DataFrame
Columns: []
Index: []


### Residualization

In [16]:
robustness_path = RAD_MAIN / "robustness" / "weight_trimming.parquet"
robustness_summary_path = RAD_MAIN / "robustness" / "weight_trimming_summary.parquet"

weight_trimming_results = pd.DataFrame()
weight_trimming_summary = pd.DataFrame()

save_df_pair(weight_trimming_results, robustness_path)
save_df_pair(weight_trimming_summary, robustness_summary_path)

weight_trimming_summary

""


### Conditional residualization

In [17]:

cond_resid_long_path = RAD_MAIN / "robustness" / "conditional_residualization_long.parquet"
cond_resid_compare_raw_path = RAD_MAIN / "robustness" / "conditional_residualization_compare_raw.parquet"
cond_resid_compare_cond_path = RAD_MAIN / "robustness" / "conditional_residualization_compare_cond.parquet"
cond_resid_shift_path = RAD_MAIN / "robustness" / "conditional_residualization_shift.parquet"
cond_resid_shift_summary_path = RAD_MAIN / "robustness" / "conditional_residualization_shift_summary.parquet"
cond_resid_r2_path = RAD_MAIN / "robustness" / "conditional_residualization_r2.parquet"

if all(
    p.exists()
    for p in [
        cond_resid_long_path,
        cond_resid_compare_raw_path,
        cond_resid_compare_cond_path,
        cond_resid_shift_path,
        cond_resid_shift_summary_path,
        cond_resid_r2_path,
    ]
):
    conditional_residualization_long = pd.read_parquet(cond_resid_long_path)
    conditional_residualization_compare_raw = pd.read_parquet(cond_resid_compare_raw_path)
    conditional_residualization_compare_cond = pd.read_parquet(cond_resid_compare_cond_path)
    conditional_residualization_shift = pd.read_parquet(cond_resid_shift_path)
    conditional_residualization_shift_summary = pd.read_parquet(cond_resid_shift_summary_path)
    conditional_residualization_r2 = pd.read_parquet(cond_resid_r2_path)
else:
    cond_long_rows = []
    cond_compare_raw_rows = []
    cond_compare_cond_rows = []
    cond_shift_rows = []
    cond_r2_rows = []

    for outcome_spec in OUTCOME_SPECS:
        time_col = outcome_spec["time_col"]
        tau = outcome_spec["tau"]
        in_risk, duration, event = build_survival_target(analytic_main, time_col, tau)
        risk_df = analytic_main.loc[in_risk].reset_index(drop=True).copy()

        train_idx, val_idx = make_train_val_split(
            len(risk_df),
            seed=stable_seed("rad_cond_resid_split", time_col),
            val_frac=VAL_FRAC,
        )

        z_cond_df, cond_meta = make_conditional_residualized_text_features(
            df=risk_df,
            text_cols=z_cols,
            structured_cols=base_adjust_cols,
            demo_cols=GROUP_VARS,
            train_idx=train_idx,
            seed=stable_seed("rad_cond_resid", time_col),
        )
        z_cond_cols = z_cond_df.columns.tolist()
        risk_aug = pd.concat([risk_df, z_cond_df], axis=1)

        cond_r2_rows.append({"outcome": time_col, **cond_meta})

        scheme_map = {
            "structured": {
                "structured_cols": GROUP_VARS + base_adjust_cols,
                "text_cols": None,
                "ipw_adjust_cols": base_adjust_cols,
                "trim_q": MAIN_TRIM_Q
            },
            "structured_plus_text": {
                "structured_cols": GROUP_VARS + base_adjust_cols,
                "text_cols": z_cols,
                "ipw_adjust_cols": text_adjust_cols,
                "trim_q": MAIN_TRIM_Q
            },
            "structured_plus_text_condresid": {
                "structured_cols": GROUP_VARS + base_adjust_cols,
                "text_cols": z_cond_cols,
                "ipw_adjust_cols": list(dict.fromkeys(base_adjust_cols + z_cond_cols)),
                "trim_q": MAIN_TRIM_Q
            },
        }

        long_cond, _, _, _, _ = fit_weighted_subset_compare(
            analytic_sub=risk_aug,
            outcome_specs=[outcome_spec],
            scheme_map=scheme_map,
            left_scheme="structured",
            right_scheme="structured_plus_text",
            seed_prefix=f"rad_cond_{time_col}",
        )
        compare_raw = compare_disparity(
            tbl_left=long_cond.loc[long_cond["scheme"] == "structured"].copy(),
            tbl_right=long_cond.loc[long_cond["scheme"] == "structured_plus_text"].copy(),
            left_scheme="structured",
            right_scheme="structured_plus_text",
        )
        compare_cond = compare_disparity(
            tbl_left=long_cond.loc[long_cond["scheme"] == "structured"].copy(),
            tbl_right=long_cond.loc[long_cond["scheme"] == "structured_plus_text_condresid"].copy(),
            left_scheme="structured",
            right_scheme="structured_plus_text_condresid",
        )
        compare_raw["outcome"] = time_col
        compare_cond["outcome"] = time_col

        shift_df = summarize_shift_attenuation(compare_raw, compare_cond, "condresid")
        shift_df["outcome"] = time_col

        cond_long_rows.append(long_cond)
        cond_compare_raw_rows.append(compare_raw)
        cond_compare_cond_rows.append(compare_cond)
        cond_shift_rows.append(shift_df)

    conditional_residualization_long = pd.concat(cond_long_rows, ignore_index=True)
    conditional_residualization_compare_raw = pd.concat(cond_compare_raw_rows, ignore_index=True)
    conditional_residualization_compare_cond = pd.concat(cond_compare_cond_rows, ignore_index=True)
    conditional_residualization_shift = pd.concat(cond_shift_rows, ignore_index=True)
    conditional_residualization_shift_summary = summarize_attenuation_table(
        conditional_residualization_shift,
        "condresid",
    )
    conditional_residualization_r2 = pd.DataFrame(cond_r2_rows)

    save_df_pair(conditional_residualization_long, cond_resid_long_path)
    save_df_pair(conditional_residualization_compare_raw, cond_resid_compare_raw_path)
    save_df_pair(conditional_residualization_compare_cond, cond_resid_compare_cond_path)
    save_df_pair(conditional_residualization_shift, cond_resid_shift_path)
    save_df_pair(conditional_residualization_shift_summary, cond_resid_shift_summary_path)
    save_df_pair(conditional_residualization_r2, cond_resid_r2_path)

conditional_residualization_shift_summary.head()


[fit_weighted_subset_compare] start seed_prefix=rad_cond_time_to_any_rad_hours
[fit_weighted_subset_compare] outcome=time_to_any_rad_hours stage=build_survival_target
[fit_weighted_subset_compare] outcome=time_to_any_rad_hours stage=risk_set n_risk=203016 n_event=117540
[fit_weighted_subset_compare] outcome=time_to_any_rad_hours family=gender scheme=structured stage=family_weight
[fit_weighted_subset_compare] outcome=time_to_any_rad_hours family=gender scheme=structured stage=model_fit
[fit_weighted_subset_compare] outcome=time_to_any_rad_hours family=gender scheme=structured_plus_text stage=family_weight
[fit_weighted_subset_compare] outcome=time_to_any_rad_hours family=gender scheme=structured_plus_text stage=model_fit
[fit_weighted_subset_compare] outcome=time_to_any_rad_hours family=gender scheme=structured_plus_text_condresid stage=family_weight
[fit_weighted_subset_compare] outcome=time_to_any_rad_hours family=gender scheme=structured_plus_text_condresid stage=model_fit
[fit_weig

,outcome,group_var,n_levels,median_abs_delta_beta_raw,median_abs_delta_beta_alt,median_alt_share_beta,median_attenuation_beta,sign_same_beta_share
0,time_to_advanced_hours,gender,2,0.017376,0.030964,1.782009,-0.782009,0.500000
1,time_to_advanced_hours,language,8,0.046004,0.097584,1.479320,-0.479320,0.625000
2,time_to_advanced_hours,race,6,0.045729,0.192559,3.051825,-2.051825,0.666667
3,time_to_any_rad_hours,gender,2,0.072553,0.056688,0.781335,0.218665,1.000000
4,time_to_any_rad_hours,language,8,0.116301,0.057684,1.145623,-0.145623,0.750000


### Transfer

In [18]:

transfer_results_path = RAD_MAIN / "robustness" / "transfer_subgroups.parquet"
transfer_counts_path = RAD_MAIN / "robustness" / "transfer_subgroups_counts.parquet"
transfer_compare_path = RAD_MAIN / "robustness" / "transfer_compare.parquet"
transfer_shift_path = RAD_MAIN / "robustness" / "transfer_shift.parquet"
transfer_shift_summary_path = RAD_MAIN / "robustness" / "transfer_shift_summary.parquet"

if all(
    p.exists()
    for p in [
        transfer_results_path,
        transfer_counts_path,
        transfer_compare_path,
        transfer_shift_path,
        transfer_shift_summary_path,
    ]
):
    transfer_results = pd.read_parquet(transfer_results_path)
    transfer_counts = pd.read_parquet(transfer_counts_path)
    transfer_compare = pd.read_parquet(transfer_compare_path)
    transfer_shift = pd.read_parquet(transfer_shift_path)
    transfer_shift_summary = pd.read_parquet(transfer_shift_summary_path)
else:
    cc_norm = normalize_cc(analytic_main[TEXT_COL])
    transfer_flag = cc_norm.str.contains("transfer", regex=False, na=False)

    subgroup_inputs = {
        "non_transfer": analytic_main.loc[~transfer_flag].reset_index(drop=True).copy(),
        "transfer": analytic_main.loc[transfer_flag].reset_index(drop=True).copy(),
    }

    transfer_long_rows = []
    transfer_compare_rows = []
    transfer_count_rows = []

    for subgroup, analytic_sub in subgroup_inputs.items():
        subgroup_long, subgroup_compare, subgroup_counts, subgroup_diag, subgroup_artifacts = fit_weighted_subset_compare(
            analytic_sub=analytic_sub,
            outcome_specs=OUTCOME_SPECS,
            scheme_map=MAIN_SCHEME_MAP,
            left_scheme="structured",
            right_scheme="structured_plus_text",
            seed_prefix=f"transfer_point_{subgroup}",
        )
        subgroup_long["subgroup"] = subgroup
        subgroup_compare["subgroup"] = subgroup
        subgroup_counts["subgroup"] = subgroup
        transfer_long_rows.append(subgroup_long)
        transfer_compare_rows.append(subgroup_compare)
        transfer_count_rows.append(subgroup_counts)

    transfer_results = pd.concat(transfer_long_rows, ignore_index=True)
    transfer_compare = pd.concat(transfer_compare_rows, ignore_index=True)
    transfer_counts = pd.concat(transfer_count_rows, ignore_index=True)
    transfer_shift, transfer_shift_summary = build_transfer_point_outputs(transfer_compare)

    save_df_pair(transfer_results, transfer_results_path)
    save_df_pair(transfer_counts, transfer_counts_path)
    save_df_pair(transfer_compare, transfer_compare_path)
    save_df_pair(transfer_shift, transfer_shift_path)
    save_df_pair(transfer_shift_summary, transfer_shift_summary_path)

transfer_shift_summary.head()


[fit_weighted_subset_compare] start seed_prefix=transfer_point_non_transfer
[fit_weighted_subset_compare] outcome=time_to_any_rad_hours stage=build_survival_target
[fit_weighted_subset_compare] outcome=time_to_any_rad_hours stage=risk_set n_risk=182468 n_event=107270
[fit_weighted_subset_compare] outcome=time_to_any_rad_hours family=gender scheme=structured stage=family_weight
[fit_weighted_subset_compare] outcome=time_to_any_rad_hours family=gender scheme=structured stage=model_fit
[fit_weighted_subset_compare] outcome=time_to_any_rad_hours family=gender scheme=structured_plus_text stage=family_weight
[fit_weighted_subset_compare] outcome=time_to_any_rad_hours family=gender scheme=structured_plus_text stage=model_fit
[fit_weighted_subset_compare] outcome=time_to_any_rad_hours family=gender stage=compare
[fit_weighted_subset_compare] outcome=time_to_any_rad_hours family=race scheme=structured stage=family_weight
[fit_weighted_subset_compare] outcome=time_to_any_rad_hours family=race sc

,outcome,group_var,n_levels,median_abs_shift_beta_non_transfer,median_abs_shift_beta_transfer,median_transfer_over_non_beta_ratio,same_direction_beta_share
0,time_to_advanced_hours,gender,2,0.041332,0.048313,1.168886,1.000000
1,time_to_advanced_hours,language,8,0.051064,0.149017,3.174858,0.750000
2,time_to_advanced_hours,race,6,0.050115,0.069328,1.578150,0.666667
3,time_to_any_rad_hours,gender,2,0.003226,0.050727,15.724052,1.000000
4,time_to_any_rad_hours,language,8,0.149652,0.137053,1.713707,0.625000


### Same chief complaint

In [24]:
same_cc_path = RAD_MAIN / "robustness" / "same_chiefcomplaint.parquet"
same_cc_term_summary_path = RAD_MAIN / "robustness" / "same_chiefcomplaint_term_summary.parquet"
same_cc_complaint_summary_path = RAD_MAIN / "robustness" / "same_chiefcomplaint_complaint_summary.parquet"
same_cc_vs_main_path = RAD_MAIN / "robustness" / "same_chiefcomplaint_vs_main.parquet"
same_cc_vs_main_summary_path = RAD_MAIN / "robustness" / "same_chiefcomplaint_vs_main_summary.parquet"
same_cc_skip_path = RAD_MAIN / "robustness" / "same_chiefcomplaint_skipped.parquet"


def _samecc_weighted_mean(x, w):
    x = np.asarray(x, dtype=float)
    w = np.asarray(w, dtype=float)
    mask = np.isfinite(x) & np.isfinite(w) & (w > 0)
    if not np.any(mask):
        return np.nan
    return float(np.sum(w[mask] * x[mask]) / np.sum(w[mask]))


def _samecc_nanquantile(x, q):
    x = np.asarray(x, dtype=float)
    x = x[np.isfinite(x)]
    if x.size == 0:
        return np.nan
    return float(np.quantile(x, q))


def _samecc_nanstat(x, fn):
    x = np.asarray(x, dtype=float)
    x = x[np.isfinite(x)]
    if x.size == 0:
        return np.nan
    return float(fn(x))


def _samecc_save_df_pair(df: pd.DataFrame, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_parquet(path, index=False)
    df.to_csv(path.with_suffix(".csv"), index=False)


def _samecc_full_compare_ref(compare_df: pd.DataFrame):
    cols = ["outcome", "group_var", "level", "ref_level", "term", "delta_beta"]
    out = compare_df.loc[:, [c for c in cols if c in compare_df.columns]].copy()
    out = out.rename(columns={"delta_beta": "delta_beta_full"})
    return out


def _samecc_weighting_input_df(df: pd.DataFrame, adjust_cols: list[str], freq_weight_col=None):
    cols = (
        list(GROUP_VARS)
        + [c for c in adjust_cols if c not in GROUP_VARS]
        + (["selection_weight"] if "selection_weight" in df.columns else [])
        + ([freq_weight_col] if freq_weight_col is not None and freq_weight_col in df.columns else [])
    )
    return df.loc[:, list(dict.fromkeys(cols))].copy()


def fit_family_overlap_weights(df, family_var, adjust_cols, seed, ref_map, freq_weight_col=None):
    d = df.copy()
    d["_row_id"] = d.index.to_numpy()
    d = d.reset_index(drop=True)

    if freq_weight_col is None:
        freq = np.ones(len(d), dtype=float)
    else:
        freq = pd.to_numeric(d[freq_weight_col], errors="coerce").to_numpy(dtype=float)

    a = d[family_var].astype("string").astype(str)
    X = standardize_features(d, [c for c in adjust_cols if c not in GROUP_VARS])

    if a.nunique() == 2:
        levels = sorted(a.unique().tolist())
        target = (a == levels[1]).astype(int).to_numpy()

        model, p1 = fit_binary_propensity(
            target,
            X,
            seed=seed,
            sample_weight=freq,
        )

        pm = np.column_stack([1.0 - p1, p1])
        prob_obs = pm[np.arange(len(d)), target]
        class_labels = levels

    else:
        levels = a.value_counts().index.tolist()
        codes = pd.Categorical(a, categories=levels).codes

        model, pm = fit_multiclass_propensity(
            codes,
            X,
            seed=seed,
            sample_weight=freq,
        )

        prob_obs = pm[np.arange(len(d)), codes]
        class_labels = levels

    overlap_h = 1.0 / np.sum(1.0 / pm, axis=1)

    d["joint_group"] = a.to_numpy()
    d["ipw_joint"] = overlap_h / prob_obs

    if "selection_weight" not in d.columns:
        d["selection_weight"] = 1.0

    d["selection_weight"] = pd.to_numeric(d["selection_weight"], errors="coerce").astype(float)
    d["joint_ref_label"] = ref_map.get(family_var)
    d["total_weight"] = d["selection_weight"] * d["ipw_joint"]

    coef_rows = []
    feature_names = list(X.columns)
    coef = np.asarray(model.coef_, dtype=float)
    intercept = np.asarray(model.intercept_, dtype=float).reshape(-1)

    if coef.ndim == 1:
        coef = coef.reshape(1, -1)

    if len(class_labels) == 2 and coef.shape[0] == 1:
        row_iter = [(class_labels[1], coef[0], intercept[0])]
    else:
        row_iter = [(class_labels[k], coef[k], intercept[k]) for k in range(len(class_labels))]

    for class_label, coef_row, intercept_val in row_iter:
        coef_rows.append({
            "diagnostic_type": "family_weight_intercept",
            "class_label": class_label,
            "feature": "(Intercept)",
            "coef": float(intercept_val),
        })
        for feature_name, coef_val in zip(feature_names, coef_row):
            coef_rows.append({
                "diagnostic_type": "family_weight_coef",
                "class_label": class_label,
                "feature": feature_name,
                "coef": float(coef_val),
            })

    q_prob = np.quantile(prob_obs, [0.01, 0.50, 0.99])
    q_weight = np.quantile(d["ipw_joint"].to_numpy(dtype=float), [0.01, 0.50, 0.99])

    diagnostics = {
        "diagnostic_type": "family_weight_fit",
        "family_var": family_var,
        "n_rows": int(len(d)),
        "n_family_levels": int(a.nunique()),
        "family_class_observed": "|".join(map(str, class_labels)),
        "family_prob_min": float(np.min(prob_obs)),
        "family_prob_p01": float(q_prob[0]),
        "family_prob_p50": float(q_prob[1]),
        "family_prob_p99": float(q_prob[2]),
        "family_prob_max": float(np.max(prob_obs)),
        "family_weight_min": float(np.min(d["ipw_joint"])),
        "family_weight_p01": float(q_weight[0]),
        "family_weight_p50": float(q_weight[1]),
        "family_weight_p99": float(q_weight[2]),
        "family_weight_max": float(np.max(d["ipw_joint"])),
    }

    return {
        "weighted_df": d,
        "diagnostics": diagnostics,
        "coef_table": pd.DataFrame(coef_rows),
        "model": model,
    }


def disparity_table_from_model_counterfactual(
    model,
    X: pd.DataFrame,
    tau: float,
    ref_map: dict,
    families=("gender_", "race_", "language_"),
    sample_weight=None,
):
    cols = X.columns.to_list()
    x_ref_template = X.copy()
    rows = []

    if sample_weight is None:
        sample_weight = np.ones(len(X), dtype=float)
    else:
        sample_weight = np.asarray(sample_weight, dtype=float)
        if len(sample_weight) != len(X):
            raise ValueError(f"sample_weight length mismatch: weight={len(sample_weight)} rows={len(X)}")

    for prefix in families:
        fam_cols = [c for c in cols if c.startswith(prefix)]
        if len(fam_cols) == 0:
            continue

        group_var = prefix[:-1]
        ref_level = ref_map.get(group_var)

        X_ref = x_ref_template.copy()
        X_ref.loc[:, fam_cols] = 0.0
        lp_ref = model.predict(X_ref)

        rows.append({
            "group_var": group_var,
            "level": ref_level,
            "ref_level": ref_level,
            "term": f"{group_var}_{ref_level}",
            "beta": 0.0,
            "HR": 1.0,
        })

        for fam_col in fam_cols:
            X_cf = X_ref.copy()
            X_cf.loc[:, fam_col] = 1.0
            lp_cf = model.predict(X_cf)

            beta = _samecc_weighted_mean(lp_cf - lp_ref, sample_weight)
            level = fam_col.split("_", 1)[1]

            rows.append({
                "group_var": group_var,
                "level": level,
                "ref_level": ref_level,
                "term": fam_col,
                "beta": beta,
                "HR": float(np.exp(beta)) if np.isfinite(beta) else np.nan,
            })

    if len(rows) == 0:
        raise ValueError(
            f"No counterfactual rows for families={families}. "
            "The fitted design matrix has no dummy columns for this family."
        )

    return pd.DataFrame(rows).sort_values(["group_var", "level"]).reset_index(drop=True)


def fit_single_scheme_model(
    risk_df: pd.DataFrame,
    duration: np.ndarray,
    event: np.ndarray,
    structured_cols: list[str],
    text_cols: list[str] | None,
    tau: float,
    seed: int,
    scheme_name: str,
    family_var: str,
    family_weight_fit: dict,
    trim_q=None,
    freq_weight_col=None,
):
    weighted_df = apply_weight_truncation(family_weight_fit["weighted_df"], trim_q=trim_q)
    row_idx = weighted_df["_row_id"].to_numpy()

    fit_df = risk_df.iloc[row_idx].reset_index(drop=True).copy()
    duration_fit = np.asarray(duration[row_idx], dtype=float)
    event_fit = np.asarray(event[row_idx], dtype=np.int8)

    if freq_weight_col is None:
        freq_weight = np.ones(len(weighted_df), dtype=float)
    else:
        freq_weight = pd.to_numeric(weighted_df[freq_weight_col], errors="coerce").to_numpy(dtype=float)

    train_freq, val_freq = split_frequency_weight(freq_weight, VAL_FRAC, seed=seed)
    train_mask = train_freq > 0
    val_mask = val_freq > 0
    train_idx = np.flatnonzero(train_mask)

    total_weight = weighted_df["total_weight"].to_numpy(dtype=float)
    total_weight_full = total_weight * freq_weight
    total_weight_train = total_weight * train_freq
    total_weight_val = total_weight * val_freq

    preproc = fit_feature_preprocessor(fit_df.iloc[train_idx].copy(), structured_cols)
    X_struct_all = transform_features(fit_df, preproc)
    demo_cols = get_demo_cols(X_struct_all)
    base_cols_no_demo = [c for c in X_struct_all.columns if c not in demo_cols]

    if text_cols is None:
        X_model = pd.concat(
            [X_struct_all[base_cols_no_demo], X_struct_all[demo_cols]],
            axis=1,
        ).astype(np.float32)
        model_kind = "structured"
        n_text = 0
        z_spec = None
    else:
        Z_std, z_spec = standardize_dense_block(fit_df, text_cols, train_idx)
        X_model = pd.concat(
            [X_struct_all[base_cols_no_demo], Z_std[text_cols], X_struct_all[demo_cols]],
            axis=1,
        ).astype(np.float32)
        model_kind = "text"
        n_text = len(text_cols)

    family_prefix = f"{family_var}_"
    family_cols = [c for c in X_model.columns if c.startswith(family_prefix)]

    if len(family_cols) == 0:
        diagnostics = {
            "diagnostic_type": "family_skipped_no_design_contrast",
            "group_var": family_var,
            "family_var": family_var,
            "scheme": scheme_name,
            "trim_label": "none" if trim_q is None else f"{trim_q[0]:.3f}_{trim_q[1]:.3f}",
            "n_rows": int(len(fit_df)),
            "n_event": int(event_fit.sum()),
            "n_demo_cols": int(len(demo_cols)),
            "n_text_cols": int(n_text),
            "skip_reason": "no_family_dummy_columns_after_training_preprocessor",
            "observed_family_levels": "|".join(
                sorted(fit_df[family_var].astype("string").astype(str).unique().tolist())
            ),
        }
        return None, diagnostics, None

    model = fit_weighted_neural_cox(
        X=X_model,
        durations=duration_fit,
        events=event_fit,
        sample_weight=total_weight_full,
        train_weight=total_weight_train,
        val_weight=total_weight_val,
        model_kind=model_kind,
        n_demo=len(demo_cols),
        n_text=n_text,
    )

    tbl = disparity_table_from_model_counterfactual(
        model=model,
        X=X_model,
        tau=tau,
        ref_map=REF,
        families=(f"{family_var}_",),
        sample_weight=total_weight_full,
    )

    tbl["scheme"] = scheme_name
    tbl["n_risk"] = int(len(fit_df))
    tbl["n_event"] = int(event_fit.sum())
    tbl["weight_sum"] = float(np.sum(total_weight_full))

    alpha_demo = model.net.alpha.weight.detach().cpu().numpy().reshape(-1)

    if model_kind == "text":
        beta_z = model.net.beta.weight.detach().cpu().numpy().reshape(-1)
        gamma = model.net.gamma.weight.detach().cpu().numpy()
    else:
        beta_z = None
        gamma = None

    cindex_train = cindex_bundle(
        model=model,
        X=X_model.loc[train_mask].copy(),
        durations=duration_fit[train_mask],
        events=event_fit[train_mask],
        sample_weight=total_weight_train[train_mask],
    )
    cindex_val = cindex_bundle(
        model=model,
        X=X_model.loc[val_mask].copy(),
        durations=duration_fit[val_mask],
        events=event_fit[val_mask],
        sample_weight=total_weight_val[val_mask],
    )

    diagnostics = {
        "diagnostic_type": "model_fit",
        "group_var": family_var,
        "family_var": family_var,
        "scheme": scheme_name,
        "trim_label": "none" if trim_q is None else f"{trim_q[0]:.3f}_{trim_q[1]:.3f}",
        "n_rows": int(len(fit_df)),
        "n_event": int(event_fit.sum()),
        "n_demo_cols": int(len(demo_cols)),
        "n_family_design_cols": int(len(family_cols)),
        "n_text_cols": int(n_text),
        "weight_min": _samecc_nanstat(total_weight_full, np.min),
        "weight_p01": _samecc_nanquantile(total_weight_full, 0.01),
        "weight_p50": _samecc_nanquantile(total_weight_full, 0.50),
        "weight_p99": _samecc_nanquantile(total_weight_full, 0.99),
        "weight_max": _samecc_nanstat(total_weight_full, np.max),
        "effective_sample_size": float((np.sum(total_weight_full) ** 2) / np.sum(total_weight_full ** 2)),
        "best_val_loss": float(model.train_info_.get("best_val_loss", np.nan)),
        "best_epoch": int(model.train_info_.get("best_epoch", -1)),
        "n_epoch_run": int(model.train_info_.get("n_epoch_run", -1)),
        "cindex_train_weighted": cindex_train["weighted"],
        "cindex_train_unweighted": cindex_train["unweighted"],
        "cindex_val_weighted": cindex_val["weighted"],
        "cindex_val_unweighted": cindex_val["unweighted"],
    }

    artifact = {
        "model": model,
        "X_model": X_model,
        "fit_df": fit_df,
        "duration": duration_fit,
        "event": event_fit,
        "sample_weight": total_weight_full,
        "demo_cols": demo_cols,
        "text_cols": [] if text_cols is None else list(text_cols),
        "alpha_demo": alpha_demo,
        "beta_z": beta_z,
        "gamma": gamma,
        "table": tbl.copy(),
        "preprocessor": preproc,
        "z_spec": z_spec,
        "family_var": family_var,
        "family_weight_fit": family_weight_fit,
    }

    return tbl, diagnostics, artifact


def fit_weighted_subset_compare(
    analytic_sub: pd.DataFrame,
    outcome_specs: list[dict],
    scheme_map: dict,
    left_scheme: str,
    right_scheme: str,
    seed_prefix: str,
    freq_weight_col=None,
):
    print(f"[fit_weighted_subset_compare] start seed_prefix={seed_prefix}")

    long_rows = []
    compare_rows = []
    count_rows = []
    diag_rows = []
    artifact_store = {}

    for outcome_spec in outcome_specs:
        time_col = outcome_spec["time_col"]
        tau = outcome_spec["tau"]

        print(f"[fit_weighted_subset_compare] outcome={time_col} stage=build_survival_target")
        in_risk, duration, event = build_survival_target(analytic_sub, time_col, tau)
        risk_df = analytic_sub.loc[in_risk].reset_index(drop=True).copy()

        print(
            f"[fit_weighted_subset_compare] outcome={time_col} stage=risk_set "
            f"n_risk={len(risk_df)} n_event={int(event.sum())}"
        )

        count_rows.append({
            "outcome": time_col,
            "tau": tau,
            "n_risk": int(len(risk_df)),
            "n_event": int(event.sum()),
        })

        artifact_store[time_col] = {}

        for family_var in GROUP_VARS:
            ref_level = str(REF[family_var])
            observed_levels = sorted(risk_df[family_var].astype("string").astype(str).unique().tolist())
            focal_levels = [x for x in observed_levels if x != ref_level]

            if ref_level not in observed_levels or len(focal_levels) == 0:
                diag_rows.append({
                    "diagnostic_type": "family_skipped_no_observed_contrast",
                    "outcome": time_col,
                    "tau": tau,
                    "group_var": family_var,
                    "family_var": family_var,
                    "scheme": "all",
                    "n_rows": int(len(risk_df)),
                    "n_event": int(event.sum()),
                    "ref_level": ref_level,
                    "observed_levels": "|".join(observed_levels),
                    "focal_levels": "|".join(focal_levels),
                    "has_reference": bool(ref_level in observed_levels),
                    "has_focal": bool(len(focal_levels) > 0),
                })
                continue

            artifact_store[time_col][family_var] = {}
            scheme_tables = []
            family_skip = False

            for scheme_name, spec in scheme_map.items():
                print(
                    f"[fit_weighted_subset_compare] outcome={time_col} "
                    f"family={family_var} scheme={scheme_name} stage=family_weight"
                )

                adjust_cols_use = [c for c in spec["ipw_adjust_cols"] if c != family_var]
                weight_input_df = _samecc_weighting_input_df(
                    risk_df,
                    adjust_cols_use,
                    freq_weight_col=freq_weight_col,
                )

                family_weight_fit = fit_family_overlap_weights(
                    weight_input_df,
                    family_var=family_var,
                    adjust_cols=adjust_cols_use,
                    seed=stable_seed(seed_prefix, time_col, family_var, scheme_name, "family_weight"),
                    ref_map=REF,
                    freq_weight_col=freq_weight_col,
                )

                family_diag = dict(family_weight_fit["diagnostics"])
                family_diag["outcome"] = time_col
                family_diag["tau"] = tau
                family_diag["scheme"] = scheme_name
                family_diag["group_var"] = family_var
                family_diag["family_var"] = family_var
                diag_rows.append(family_diag)

                coef_tbl = family_weight_fit["coef_table"].copy()
                if len(coef_tbl) > 0:
                    coef_tbl["outcome"] = time_col
                    coef_tbl["tau"] = tau
                    coef_tbl["scheme"] = scheme_name
                    coef_tbl["group_var"] = family_var
                    coef_tbl["family_var"] = family_var
                    diag_rows.extend(coef_tbl.to_dict("records"))

                print(
                    f"[fit_weighted_subset_compare] outcome={time_col} "
                    f"family={family_var} scheme={scheme_name} stage=model_fit"
                )

                tbl, diag, artifact = fit_single_scheme_model(
                    risk_df=risk_df,
                    duration=duration,
                    event=event,
                    structured_cols=spec["structured_cols"],
                    text_cols=spec.get("text_cols"),
                    tau=tau,
                    seed=stable_seed(seed_prefix, time_col, family_var, scheme_name, "family_model"),
                    scheme_name=scheme_name,
                    family_var=family_var,
                    family_weight_fit=family_weight_fit,
                    trim_q=spec.get("trim_q"),
                    freq_weight_col=freq_weight_col,
                )

                diag["outcome"] = time_col
                diag["tau"] = tau
                diag["group_var"] = family_var
                diag["family_var"] = family_var
                diag_rows.append(diag)

                if tbl is None:
                    family_skip = True
                    break

                tbl["outcome"] = time_col
                tbl["tau"] = tau
                tbl["family_var"] = family_var
                scheme_tables.append(tbl)
                artifact_store[time_col][family_var][scheme_name] = artifact

            if family_skip:
                artifact_store[time_col].pop(family_var, None)
                continue

            if len(scheme_tables) != 2:
                artifact_store[time_col].pop(family_var, None)
                continue

            print(f"[fit_weighted_subset_compare] outcome={time_col} family={family_var} stage=compare")

            outcome_long = pd.concat(scheme_tables, ignore_index=True)
            long_rows.append(outcome_long)

            cmp = compare_disparity(
                tbl_left=outcome_long.loc[outcome_long["scheme"] == left_scheme].copy(),
                tbl_right=outcome_long.loc[outcome_long["scheme"] == right_scheme].copy(),
                left_scheme=left_scheme,
                right_scheme=right_scheme,
            )
            cmp["outcome"] = time_col
            cmp["tau"] = tau
            cmp["family_var"] = family_var
            compare_rows.append(cmp)

        print(f"[fit_weighted_subset_compare] outcome={time_col} stage=done")

    print(f"[fit_weighted_subset_compare] done seed_prefix={seed_prefix}")

    long_df = pd.concat(long_rows, ignore_index=True) if len(long_rows) else pd.DataFrame()
    compare_df = pd.concat(compare_rows, ignore_index=True) if len(compare_rows) else pd.DataFrame()
    counts_df = pd.DataFrame(count_rows)
    diag_df = pd.DataFrame(diag_rows)

    return long_df, compare_df, counts_df, diag_df, artifact_store


if all(
    p.exists()
    for p in [
        same_cc_path,
        same_cc_term_summary_path,
        same_cc_complaint_summary_path,
        same_cc_vs_main_path,
        same_cc_vs_main_summary_path,
    ]
):
    same_cc_results = pd.read_parquet(same_cc_path)
    same_cc_term_summary = pd.read_parquet(same_cc_term_summary_path)
    same_cc_complaint_summary = pd.read_parquet(same_cc_complaint_summary_path)
    same_cc_vs_main = pd.read_parquet(same_cc_vs_main_path)
    same_cc_vs_main_summary = pd.read_parquet(same_cc_vs_main_summary_path)
    same_cc_skipped = pd.read_parquet(same_cc_skip_path) if same_cc_skip_path.exists() else pd.DataFrame()

else:
    rows = []
    skipped_rows = []

    d = analytic_main.loc[analytic_main[TEXT_COL] != "no_cc"].copy()
    cc_counts = d[TEXT_COL].value_counts()
    keep_cc = cc_counts.index[cc_counts >= 50]
    d = d.loc[d[TEXT_COL].isin(keep_cc)].reset_index(drop=True)

    for outcome_spec in OUTCOME_SPECS:
        time_col = outcome_spec["time_col"]
        tau = outcome_spec["tau"]

        in_risk, duration, event = build_survival_target(d, time_col, tau)
        risk_df = d.loc[in_risk].reset_index(drop=True).copy()
        event = np.asarray(event, dtype=int)

        for cc, idx in risk_df.groupby(TEXT_COL).groups.items():
            idx = np.asarray(list(idx), dtype=int)
            analytic_sub = risk_df.iloc[idx].reset_index(drop=True).copy()
            sub_event_n = int(event[idx].sum())

            if len(analytic_sub) < MIN_SUBSET_N:
                skipped_rows.append({
                    "outcome": time_col,
                    "chiefcomplaint": cc,
                    "skip_reason": "subset_n_below_minimum",
                    "n_rows": int(len(analytic_sub)),
                    "n_event": sub_event_n,
                })
                continue

            if sub_event_n < MIN_SUBSET_EVENT:
                skipped_rows.append({
                    "outcome": time_col,
                    "chiefcomplaint": cc,
                    "skip_reason": "subset_event_below_minimum",
                    "n_rows": int(len(analytic_sub)),
                    "n_event": sub_event_n,
                })
                continue

            subgroup_long, subgroup_compare, subgroup_counts, subgroup_diag, subgroup_artifacts = fit_weighted_subset_compare(
                analytic_sub=analytic_sub,
                outcome_specs=[outcome_spec],
                scheme_map=MAIN_SCHEME_MAP,
                left_scheme="structured",
                right_scheme="structured_plus_text",
                seed_prefix=f"samecc_{cc}_{time_col}",
            )

            if len(subgroup_compare) == 0:
                skipped_rows.append({
                    "outcome": time_col,
                    "chiefcomplaint": cc,
                    "skip_reason": "no_estimable_family_after_design_check",
                    "n_rows": int(len(analytic_sub)),
                    "n_event": sub_event_n,
                })
                if len(subgroup_diag) > 0:
                    tmp = subgroup_diag.copy()
                    tmp["chiefcomplaint"] = cc
                    skipped_rows.extend(tmp.to_dict("records"))
                continue

            subgroup_compare["chiefcomplaint"] = cc
            rows.append(subgroup_compare)

            if len(subgroup_diag) > 0:
                tmp = subgroup_diag.copy()
                tmp["chiefcomplaint"] = cc
                tmp["skip_reason"] = "not_skipped"
                skipped_rows.extend(tmp.to_dict("records"))

    same_cc_results = pd.concat(rows, ignore_index=True) if len(rows) else pd.DataFrame()
    same_cc_skipped = pd.DataFrame(skipped_rows)

    if len(same_cc_results) == 0:
        same_cc_term_summary = pd.DataFrame()
        same_cc_complaint_summary = pd.DataFrame()
        same_cc_vs_main = pd.DataFrame()
        same_cc_vs_main_summary = pd.DataFrame()

    else:
        same_cc_term_summary = (
            same_cc_results
            .groupby(["outcome", "group_var", "level", "ref_level", "term"], as_index=False)
            .agg(
                n_complaints=("chiefcomplaint", "nunique"),
                median_delta_beta=("delta_beta", "median"),
                median_abs_delta_beta=("delta_beta", lambda x: np.median(np.abs(x))),
                min_delta_beta=("delta_beta", "min"),
                max_delta_beta=("delta_beta", "max"),
                same_direction_share=("delta_beta", lambda x: max(np.mean(np.asarray(x) > 0), np.mean(np.asarray(x) < 0))),
            )
        )

        same_cc_complaint_summary = (
            same_cc_results
            .groupby(["chiefcomplaint", "outcome", "group_var"], as_index=False)
            .agg(
                n_terms=("term", "size"),
                median_abs_delta_beta=("delta_beta", lambda x: np.median(np.abs(x))),
                max_abs_delta_beta=("delta_beta", lambda x: np.max(np.abs(x))),
            )
            .sort_values(["outcome", "group_var", "median_abs_delta_beta"], ascending=[True, True, False])
            .reset_index(drop=True)
        )

        full_compare_ref = _samecc_full_compare_ref(compare_results)

        same_cc_vs_main = same_cc_term_summary.merge(
            full_compare_ref,
            on=["outcome", "group_var", "level", "ref_level", "term"],
            how="inner",
        )

        same_cc_vs_main["samecc_over_main_beta_ratio"] = (
            same_cc_vs_main["median_abs_delta_beta"] / same_cc_vs_main["delta_beta_full"].abs()
        )
        same_cc_vs_main.loc[
            same_cc_vs_main["delta_beta_full"].abs() == 0.0,
            "samecc_over_main_beta_ratio",
        ] = np.nan

        same_cc_vs_main["complaint_mix_attenuation_beta"] = (
            1.0 - same_cc_vs_main["samecc_over_main_beta_ratio"]
        )

        same_cc_vs_main["same_direction_main_vs_samecc_beta"] = (
            np.sign(same_cc_vs_main["median_delta_beta"])
            == np.sign(same_cc_vs_main["delta_beta_full"])
        )

        same_cc_vs_main_summary = (
            same_cc_vs_main
            .groupby(["outcome", "group_var"], as_index=False)
            .agg(
                n_terms=("term", "size"),
                median_samecc_over_main_beta_ratio=("samecc_over_main_beta_ratio", "median"),
                median_complaint_mix_attenuation_beta=("complaint_mix_attenuation_beta", "median"),
                share_same_direction_beta=("same_direction_main_vs_samecc_beta", "mean"),
            )
        )

    _samecc_save_df_pair(same_cc_results, same_cc_path)
    _samecc_save_df_pair(same_cc_term_summary, same_cc_term_summary_path)
    _samecc_save_df_pair(same_cc_complaint_summary, same_cc_complaint_summary_path)
    _samecc_save_df_pair(same_cc_vs_main, same_cc_vs_main_path)
    _samecc_save_df_pair(same_cc_vs_main_summary, same_cc_vs_main_summary_path)
    _samecc_save_df_pair(same_cc_skipped, same_cc_skip_path)

same_cc_vs_main_summary.head()

[fit_weighted_subset_compare] start seed_prefix=samecc_ABD PAIN_time_to_any_rad_hours
[fit_weighted_subset_compare] outcome=time_to_any_rad_hours stage=build_survival_target
[fit_weighted_subset_compare] outcome=time_to_any_rad_hours stage=risk_set n_risk=2602 n_event=1621
[fit_weighted_subset_compare] outcome=time_to_any_rad_hours family=gender scheme=structured stage=family_weight
[fit_weighted_subset_compare] outcome=time_to_any_rad_hours family=gender scheme=structured stage=model_fit
[fit_weighted_subset_compare] outcome=time_to_any_rad_hours family=gender scheme=structured_plus_text stage=family_weight
[fit_weighted_subset_compare] outcome=time_to_any_rad_hours family=gender scheme=structured_plus_text stage=model_fit
[fit_weighted_subset_compare] outcome=time_to_any_rad_hours family=gender stage=compare
[fit_weighted_subset_compare] outcome=time_to_any_rad_hours family=race scheme=structured stage=family_weight
[fit_weighted_subset_compare] outcome=time_to_any_rad_hours family=r

,outcome,group_var,n_terms,median_samecc_over_main_beta_ratio,median_complaint_mix_attenuation_beta,share_same_direction_beta
0,time_to_advanced_hours,gender,2,2.021699,-1.021699,0.500
1,time_to_advanced_hours,language,8,1.670126,-0.670126,0.500
2,time_to_advanced_hours,race,6,4.681643,-3.681643,0.500
3,time_to_any_rad_hours,gender,2,2.451079,-1.451079,1.000
4,time_to_any_rad_hours,language,8,1.872194,-0.872194,0.625


### Objective complaint

In [25]:

OBJECTIVE_COMPLAINTS = [
    "chest pain",
    "vaginal bleeding",
    "hypoglycemia",
    "hyperglycemia",
    "hematuria",
    "gunshot wound",
    "cell crisis",
    "clotted fistula",
    "hypertension crisis",
    "abnormal labs",
]

objective_compare_path = RAD_MAIN / "robustness" / "objective_complaint_compare.parquet"
objective_counts_path = RAD_MAIN / "robustness" / "objective_complaint_counts.parquet"
objective_vs_full_path = RAD_MAIN / "robustness" / "objective_complaint_vs_full.parquet"
objective_summary_path = RAD_MAIN / "robustness" / "objective_complaint_summary.parquet"

if all(p.exists() for p in [objective_compare_path, objective_counts_path, objective_vs_full_path, objective_summary_path]):
    objective_complaint_compare = pd.read_parquet(objective_compare_path)
    objective_complaint_counts = pd.read_parquet(objective_counts_path)
    objective_complaint_vs_full = pd.read_parquet(objective_vs_full_path)
    objective_complaint_summary = pd.read_parquet(objective_summary_path)
else:
    cc_norm = normalize_cc(analytic_main[TEXT_COL])
    objective_mask = cc_norm.isin(OBJECTIVE_COMPLAINTS)
    analytic_objective = analytic_main.loc[objective_mask].reset_index(drop=True).copy()

    _long_objective, objective_complaint_compare, objective_complaint_counts, objective_diag, objective_artifacts = fit_weighted_subset_compare(
        analytic_sub=analytic_objective,
        outcome_specs=OUTCOME_SPECS,
        scheme_map=MAIN_SCHEME_MAP,
        left_scheme="structured",
        right_scheme="structured_plus_text",
        seed_prefix="objective_complaint_point",
    )

    full_compare_ref = get_full_compare_ref(compare_results)
    objective_complaint_vs_full, objective_complaint_summary = build_objective_point_outputs(
        objective_compare=objective_complaint_compare,
        objective_counts=objective_complaint_counts,
        full_compare_ref=full_compare_ref,
    )

    save_df_pair(objective_complaint_compare, objective_compare_path)
    save_df_pair(objective_complaint_counts, objective_counts_path)
    save_df_pair(objective_complaint_vs_full, objective_vs_full_path)
    save_df_pair(objective_complaint_summary, objective_summary_path)

objective_complaint_summary.head()


[fit_weighted_subset_compare] start seed_prefix=objective_complaint_point
[fit_weighted_subset_compare] outcome=time_to_any_rad_hours stage=build_survival_target
[fit_weighted_subset_compare] outcome=time_to_any_rad_hours stage=risk_set n_risk=11464 n_event=7866
[fit_weighted_subset_compare] outcome=time_to_any_rad_hours family=gender scheme=structured stage=family_weight
[fit_weighted_subset_compare] outcome=time_to_any_rad_hours family=gender scheme=structured stage=model_fit
[fit_weighted_subset_compare] outcome=time_to_any_rad_hours family=gender scheme=structured_plus_text stage=family_weight
[fit_weighted_subset_compare] outcome=time_to_any_rad_hours family=gender scheme=structured_plus_text stage=model_fit
[fit_weighted_subset_compare] outcome=time_to_any_rad_hours family=gender stage=compare
[fit_weighted_subset_compare] outcome=time_to_any_rad_hours family=race scheme=structured stage=family_weight
[fit_weighted_subset_compare] outcome=time_to_any_rad_hours family=race scheme=

,outcome,group_var,n_levels,median_abs_shift_objective_beta,median_abs_shift_full_beta,median_objective_over_full_beta_ratio,same_direction_beta_vs_full_share,tau,n_risk,n_event
0,time_to_advanced_hours,gender,2,0.007817,0.038104,0.205163,0.500000,6.0,11464,1584
1,time_to_advanced_hours,language,8,0.102558,0.101112,2.302685,0.750000,6.0,11464,1584
2,time_to_advanced_hours,race,6,0.052636,0.032400,2.446833,0.666667,6.0,11464,1584
3,time_to_any_rad_hours,gender,2,0.024380,0.030936,0.788087,1.000000,6.0,11464,7866
4,time_to_any_rad_hours,language,8,0.102146,0.081430,1.230123,0.750000,6.0,11464,7866


### Complaint heterogeneity

In [26]:

INTERACTION_COMPLAINTS = [
    "chest pain",
    "dyspnea",
    "fever",
    "weakness",
    "abdominal pain"
]

complaint_heterogeneity_compare_path = RAD_MAIN / "robustness" / "complaint_heterogeneity_compare.parquet"
complaint_heterogeneity_counts_path = RAD_MAIN / "robustness" / "complaint_heterogeneity_counts.parquet"
complaint_pattern_detail_path = RAD_MAIN / "robustness" / "complaint_heterogeneity_detail.parquet"
complaint_pattern_summary_path = RAD_MAIN / "robustness" / "complaint_heterogeneity_summary.parquet"
complaint_pattern_top_diff_path = RAD_MAIN / "robustness" / "complaint_heterogeneity_top_diff.parquet"

if all(
    p.exists()
    for p in [
        complaint_heterogeneity_compare_path,
        complaint_heterogeneity_counts_path,
        complaint_pattern_detail_path,
        complaint_pattern_summary_path,
        complaint_pattern_top_diff_path,
    ]
):
    complaint_heterogeneity_compare = pd.read_parquet(complaint_heterogeneity_compare_path)
    complaint_heterogeneity_counts = pd.read_parquet(complaint_heterogeneity_counts_path)
    complaint_pattern_detail = pd.read_parquet(complaint_pattern_detail_path)
    complaint_pattern_summary = pd.read_parquet(complaint_pattern_summary_path)
    complaint_pattern_top_diff = pd.read_parquet(complaint_pattern_top_diff_path)
else:
    cc_norm = normalize_cc(analytic_main[TEXT_COL])

    compare_rows = []
    count_rows = []

    for complaint in tqdm(INTERACTION_COMPLAINTS, desc="Complaint heterogeneity", leave=True, dynamic_ncols=True):
        flag = cc_norm.eq(complaint)
        subgroup_inputs = {
            "negative": analytic_main.loc[~flag].reset_index(drop=True).copy(),
            "positive": analytic_main.loc[flag].reset_index(drop=True).copy(),
        }

        for subgroup, analytic_sub in subgroup_inputs.items():
            _, subgroup_compare, subgroup_counts, subgroup_diag, subgroup_artifacts = fit_weighted_subset_compare(
                analytic_sub=analytic_sub,
                outcome_specs=OUTCOME_SPECS,
                scheme_map=MAIN_SCHEME_MAP,
                left_scheme="structured",
                right_scheme="structured_plus_text",
                seed_prefix=f"complaint_{complaint}_{subgroup}",
            )
            subgroup_compare["complaint"] = complaint
            subgroup_compare["subgroup"] = subgroup
            subgroup_counts["complaint"] = complaint
            subgroup_counts["subgroup"] = subgroup
            compare_rows.append(subgroup_compare)
            count_rows.append(subgroup_counts)

    complaint_heterogeneity_compare = pd.concat(compare_rows, ignore_index=True)
    complaint_heterogeneity_counts = pd.concat(count_rows, ignore_index=True)

    complaint_pattern_detail = (
        complaint_heterogeneity_compare[[
            "complaint",
            "outcome",
            "group_var",
            "level",
            "ref_level",
            "term",
            "subgroup",
            "delta_beta",
        ]]
        .pivot(
            index=["complaint", "outcome", "group_var", "level", "ref_level", "term"],
            columns="subgroup",
            values=["delta_beta"],
        )
        .reset_index()
    )
    complaint_pattern_detail = flatten_columns(complaint_pattern_detail)
    complaint_pattern_detail["same_direction_beta"] = (
        np.sign(complaint_pattern_detail["delta_beta_positive"])
        == np.sign(complaint_pattern_detail["delta_beta_negative"])
    )
    complaint_pattern_detail["abs_gap_delta_beta"] = np.abs(
        complaint_pattern_detail["delta_beta_positive"]
        - complaint_pattern_detail["delta_beta_negative"]
    )

    complaint_pattern_summary = (
        complaint_pattern_detail
        .groupby(["complaint", "outcome", "group_var"], as_index=False)
        .agg(
            n_terms=("term", "size"),
            same_direction_share=("same_direction_beta", "mean"),
            median_abs_gap_delta_beta=("abs_gap_delta_beta", "median"),
        )
    )

    complaint_pattern_top_diff = (
        complaint_pattern_detail
        .sort_values(["complaint", "outcome", "group_var", "abs_gap_delta_beta"], ascending=[True, True, True, False])
        .groupby(["complaint", "outcome", "group_var"], as_index=False)
        .head(12)
        .reset_index(drop=True)
    )

    save_df_pair(complaint_heterogeneity_compare, complaint_heterogeneity_compare_path)
    save_df_pair(complaint_heterogeneity_counts, complaint_heterogeneity_counts_path)
    save_df_pair(complaint_pattern_detail, complaint_pattern_detail_path)
    save_df_pair(complaint_pattern_summary, complaint_pattern_summary_path)
    save_df_pair(complaint_pattern_top_diff, complaint_pattern_top_diff_path)

complaint_pattern_summary.head()


Complaint heterogeneity:   0%|          | 0/5 [00:00<?, ?it/s]

[fit_weighted_subset_compare] start seed_prefix=complaint_chest pain_negative
[fit_weighted_subset_compare] outcome=time_to_any_rad_hours stage=build_survival_target
[fit_weighted_subset_compare] outcome=time_to_any_rad_hours stage=risk_set n_risk=195302 n_event=111625
[fit_weighted_subset_compare] outcome=time_to_any_rad_hours family=gender scheme=structured stage=family_weight
[fit_weighted_subset_compare] outcome=time_to_any_rad_hours family=gender scheme=structured stage=model_fit
[fit_weighted_subset_compare] outcome=time_to_any_rad_hours family=gender scheme=structured_plus_text stage=family_weight
[fit_weighted_subset_compare] outcome=time_to_any_rad_hours family=gender scheme=structured_plus_text stage=model_fit
[fit_weighted_subset_compare] outcome=time_to_any_rad_hours family=gender stage=compare
[fit_weighted_subset_compare] outcome=time_to_any_rad_hours family=race scheme=structured stage=family_weight
[fit_weighted_subset_compare] outcome=time_to_any_rad_hours family=race 

Complaint heterogeneity:  20%|██        | 1/5 [1:15:26<5:01:45, 4526.33s/it]

[fit_weighted_subset_compare] start seed_prefix=complaint_dyspnea_negative
[fit_weighted_subset_compare] outcome=time_to_any_rad_hours stage=build_survival_target
[fit_weighted_subset_compare] outcome=time_to_any_rad_hours stage=risk_set n_risk=197449 n_event=113047
[fit_weighted_subset_compare] outcome=time_to_any_rad_hours family=gender scheme=structured stage=family_weight
[fit_weighted_subset_compare] outcome=time_to_any_rad_hours family=gender scheme=structured stage=model_fit
[fit_weighted_subset_compare] outcome=time_to_any_rad_hours family=gender scheme=structured_plus_text stage=family_weight
[fit_weighted_subset_compare] outcome=time_to_any_rad_hours family=gender scheme=structured_plus_text stage=model_fit
[fit_weighted_subset_compare] outcome=time_to_any_rad_hours family=gender stage=compare
[fit_weighted_subset_compare] outcome=time_to_any_rad_hours family=race scheme=structured stage=family_weight
[fit_weighted_subset_compare] outcome=time_to_any_rad_hours family=race sch

Complaint heterogeneity:  40%|████      | 2/5 [2:39:12<4:01:00, 4820.15s/it]

...

[fit_weighted_subset_compare] start seed_prefix=complaint_weakness_negative
[fit_weighted_subset_compare] outcome=time_to_any_rad_hours stage=build_survival_target
[fit_weighted_subset_compare] outcome=time_to_any_rad_hours stage=risk_set n_risk=200809 n_event=115894
[fit_weighted_subset_compare] outcome=time_to_any_rad_hours family=gender scheme=structured stage=family_weight
[fit_weighted_subset_compare] outcome=time_to_any_rad_hours family=gender scheme=structured stage=model_fit
[fit_weighted_subset_compare] outcome=time_to_any_rad_hours family=gender scheme=structured_plus_text stage=family_weight
[fit_weighted_subset_compare] outcome=time_to_any_rad_hours family=gender scheme=structured_plus_text stage=model_fit
[fit_weighted_subset_compare] outcome=time_to_any_rad_hours family=gender stage=compare
[fit_weighted_subset_compare] outcome=time_to_any_rad_hours family=race scheme=structured stage=family_weight
[fit_weighted_subset_compare] outcome=time_to_any_rad_hours family=race sc

Complaint heterogeneity:  80%|████████  | 4/5 [5:11:36<1:17:35, 4655.01s/it]

[fit_weighted_subset_compare] start seed_prefix=complaint_abdominal pain_negative
[fit_weighted_subset_compare] outcome=time_to_any_rad_hours stage=build_survival_target
[fit_weighted_subset_compare] outcome=time_to_any_rad_hours stage=risk_set n_risk=201983 n_event=116906
[fit_weighted_subset_compare] outcome=time_to_any_rad_hours family=gender scheme=structured stage=family_weight
[fit_weighted_subset_compare] outcome=time_to_any_rad_hours family=gender scheme=structured stage=model_fit
[fit_weighted_subset_compare] outcome=time_to_any_rad_hours family=gender scheme=structured_plus_text stage=family_weight
[fit_weighted_subset_compare] outcome=time_to_any_rad_hours family=gender scheme=structured_plus_text stage=model_fit
[fit_weighted_subset_compare] outcome=time_to_any_rad_hours family=gender stage=compare
[fit_weighted_subset_compare] outcome=time_to_any_rad_hours family=race scheme=structured stage=family_weight
[fit_weighted_subset_compare] outcome=time_to_any_rad_hours family=r

Complaint heterogeneity: 100%|██████████| 5/5 [6:27:41<00:00, 4652.27s/it]  


,complaint,outcome,group_var,n_terms,same_direction_share,median_abs_gap_delta_beta
0,abdominal pain,time_to_advanced_hours,gender,2,0.500,0.057913
1,abdominal pain,time_to_advanced_hours,language,8,0.375,0.348389
2,abdominal pain,time_to_advanced_hours,race,6,0.500,0.197968
3,abdominal pain,time_to_any_rad_hours,gender,2,1.000,0.067228
4,abdominal pain,time_to_any_rad_hours,language,8,0.250,0.194437
